# 🕸️ GraphRAG on Cloud Spanner — vectors, full-text and a property graph in one database

The companion to `1.ipynb`. Same idea, different engine: instead of bolting a graph onto
Postgres with recursive CTEs, this uses **Spanner Graph**, where the graph is a first-class
citizen with its own query language.

```
question
   ├─ hybrid search      COSINE_DISTANCE  +  SEARCH()/SCORE()   ← one SQL statement, RRF-fused
   ├─ entity linking     which nodes do those chunks mention?
   ├─ graph expansion    GQL:  MATCH (a)-[:Rel]->{1,2}(b)       ← this is the part that changes
   └─ generation         passages + subgraph facts → Gemini
```

## What is actually different from the Postgres version

| | Postgres (`1.ipynb`) | Spanner (this file) |
|---|---|---|
| Graph traversal | recursive CTE, hand-written | **GQL** — `MATCH (a)-[:Rel]->{1,2}(b)` |
| Shortest path | write it yourself | `MATCH ANY SHORTEST (a)-[:Rel]->{1,5}(b)` |
| Vector search | pgvector `<=>` + HNSW | `COSINE_DISTANCE` / `APPROX_COSINE_DISTANCE` + vector index |
| Keyword search | `tsvector` + GIN | `TOKENLIST` + `SEARCH()` / `SCORE()` |
| Scale ceiling | one machine | horizontal, to trillions of vectors |
| Cost floor | you already pay for it | a new Spanner instance |

**Spanner Graph's real advantage is expressiveness, not speed at your current size.** A 3-hop
path query with a cost function is one line of GQL and forty lines of recursive CTE. If your
graph stays small, Postgres is cheaper and fine. If graph queries become the main workload,
this is where you land.

---

## ⚠️ Prerequisites — read these three, they will each bite you

1. **GoogleSQL dialect.** Spanner Graph *"isn't available in the PostgreSQL dialect."* When
   you create the database, do **not** pick the PostgreSQL interface. This is chosen at
   creation and cannot be changed afterwards.
2. **Enterprise edition or higher.** Spanner Graph, vector search and vector indexes all state
   *"available with the Spanner Enterprise edition and Enterprise Plus edition."* Standard
   edition will fail at the `CREATE PROPERTY GRAPH` step.
3. **Cost.** Spanner bills for provisioned capacity whether or not you query it. This notebook
   works on the smallest unit (100 processing units); delete the database when you're done —
   there's a teardown cell at the end.

Sources: [Spanner Graph overview](https://docs.cloud.google.com/spanner/docs/graph/overview) ·
[schema statements](https://docs.cloud.google.com/spanner/docs/reference/standard-sql/graph-schema-statements) ·
[graph queries](https://docs.cloud.google.com/spanner/docs/reference/standard-sql/graph-query-statements) ·
[vector search](https://docs.cloud.google.com/spanner/docs/find-k-nearest-neighbors) ·
[full-text search](https://docs.cloud.google.com/spanner/docs/full-text-search)

## 1 — Install

In [ ]:
%pip install -q google-cloud-spanner google-genai tiktoken pandas

print("Installed. Restart the kernel if pip asked you to.")

## 2 — Configuration

**Authentication:** `gcloud auth application-default login`, or a service account with
`roles/spanner.databaseAdmin` (to create the schema) and `roles/aiplatform.user` (for Gemini).

In [ ]:
# =============================== Spanner ===============================
PROJECT_ID   = "div-aais-rfpiq-usc1-uat"   # ⬅️ your GCP project
INSTANCE_ID  = "graphrag-demo"             # ⬅️ an EXISTING Spanner instance (Enterprise edition)
DATABASE_ID  = "procurement"               # created for you by cell 3 if missing
GRAPH_NAME   = "ProcurementGraph"          # the property graph declared in cell 4

# =============================== Vertex AI =============================
LOCATION     = "us-central1"
EMBED_MODEL  = "gemini-embedding-001"
EMBED_DIM    = 1536
CHAT_MODEL   = "gemini-2.5-flash"
RERANK_MODEL = "gemini-2.5-flash"

# =============================== Chunking ==============================
CHUNK_TOKENS         = 350
CHUNK_OVERLAP_TOKENS = 60
MAX_EMBED_TOKENS     = 2000

# =============================== Retrieval =============================
VECTOR_CANDIDATES = 40    # vector leg depth
TEXT_CANDIDATES   = 40    # keyword leg depth
RRF_K             = 60    # Reciprocal Rank Fusion constant
RERANK_CANDIDATES = 20
RERANK_TOP_N      = 4
MAX_COSINE_DISTANCE = 0.80

# =============================== Graph =================================
GRAPH_HOPS             = 2    # how far to walk from the entities the question landed on
GRAPH_HUB_DEGREE_MAX   = 60   # never RELAY through a node this connected (it can still be an answer)
GRAPH_MAX_NODES        = 60
GRAPH_MAX_EXTRA_CHUNKS = 12
GRAPH_MAX_CHUNKS_PER_DOC = 2  # diversity guard — stops one long document eating the budget
GRAPH_SEED_CHUNKS      = 6
GRAPH_MAX_FACTS        = 40

# Set True only if you have >100k chunks. An ANN index brings strict query requirements
# (see cell 4b) and buys nothing at small scale, where exact KNN is already fast.
USE_VECTOR_INDEX = False

print(f"Config: {PROJECT_ID}/{INSTANCE_ID}/{DATABASE_ID} | graph={GRAPH_NAME} | "
      f"embed={EMBED_MODEL}@{EMBED_DIM}d")

## 3 — Connect

Three things to know about the Spanner Python client, because they differ from psycopg2:

- **DDL is asynchronous.** `update_ddl()` returns an operation; you must `.result()` on it.
- **Reads use a snapshot**, not a connection: `with database.snapshot() as snap: snap.execute_sql(...)`.
- **Writes are either mutations** (`database.batch()` — fast, no SQL) **or DML**
  (`run_in_transaction`). Bulk loading uses mutations.

In [ ]:
from google.cloud import spanner
from google.cloud.spanner_v1 import param_types
from google.api_core import exceptions as gexc
import pandas as pd
import json, math, re, time, random, hashlib

spanner_client = spanner.Client(project=PROJECT_ID)
instance = spanner_client.instance(INSTANCE_ID)

if not instance.exists():
    raise SystemExit(
        f"Instance '{INSTANCE_ID}' not found in project {PROJECT_ID}.\n"
        "  Create one (Enterprise edition, 100 processing units is enough):\n"
        f"    gcloud spanner instances create {INSTANCE_ID} \\\n"
        f"        --config=regional-us-central1 --description='GraphRAG demo' \\\n"
        "        --processing-units=100 --edition=ENTERPRISE")

database = instance.database(DATABASE_ID)
if not database.exists():
    print(f"Creating database '{DATABASE_ID}' (GoogleSQL dialect)…")
    # No database_dialect argument => GoogleSQL, which is what Spanner Graph requires.
    database.create().result(300)
    print("   created.")

def run_ddl(statements, timeout=600):
    """Apply DDL. Spanner batches statements into one schema change — cheaper than one call each."""
    op = database.update_ddl([s for s in statements if s.strip()])
    op.result(timeout)

def query(sql, params=None, types=None):
    """Run a read query. Returns a list of dicts."""
    with database.snapshot() as snap:
        rs = snap.execute_sql(sql, params=params, param_types=types)
        cols = [f.name for f in rs.fields]
        return [dict(zip(cols, row)) for row in rs]

def query_df(sql, params=None, types=None):
    return pd.DataFrame(query(sql, params, types))

def write(table, columns, values, chunk=500):
    """Bulk insert-or-update via mutations. Idempotent on primary key."""
    for i in range(0, len(values), chunk):
        with database.batch() as batch:
            batch.insert_or_update(table=table, columns=columns, values=values[i:i + chunk])

_v = query("SELECT 1 AS ok")
print(f"Connected to {PROJECT_ID}/{INSTANCE_ID}/{DATABASE_ID} — ok={_v[0]['ok']}")

## 4 — Schema

Four tables and one property graph.

`Documents` / `Chunks` are the RAG side. `GraphNode` / `GraphEdge` are the graph side, and
`NodeMention` is the bridge between them — *this node is talked about in that chunk*. Without
that bridge you have a graph and a vector store that never meet.

**Two deliberate choices worth understanding:**

**Embeddings are `ARRAY<FLOAT64>`, not `FLOAT32`.** Python floats map to `FLOAT64` natively, so
mutations insert without any casting. `FLOAT32` halves storage and is what the vector-index
docs use — switch to it if you add an index at scale, but then you must cast on write.

**Every edge is stored twice, forward and reverse** (`is_reverse`). Traversal is then always
`->` and never needs the undirected form, which keeps every GQL pattern in this notebook to
syntax that is documented with a worked example. `WHERE NOT e.is_reverse` recovers the real
direction whenever you care.

In [ ]:
DDL = [
f"""CREATE TABLE IF NOT EXISTS Documents (
  doc_id        STRING(128) NOT NULL,
  title         STRING(MAX) NOT NULL,
  content       STRING(MAX) NOT NULL,
  metadata      JSON,
  content_hash  STRING(64),
  indexed_hash  STRING(64),
  updated_at    TIMESTAMP OPTIONS (allow_commit_timestamp=true),
) PRIMARY KEY (doc_id)""",

f"""CREATE TABLE IF NOT EXISTS Chunks (
  doc_id       STRING(128) NOT NULL,
  chunk_index  INT64 NOT NULL,
  chunk_id     STRING(160) NOT NULL,          -- doc_id#index, so it is stable across re-ingest
  content      STRING(MAX) NOT NULL,          -- clean text, shown to the model and to you
  embed_input  STRING(MAX) NOT NULL,          -- header + text: what we embed AND tokenize
  token_count  INT64,
  metadata     JSON,
  embedding    ARRAY<FLOAT64>(vector_length=>{EMBED_DIM}),
  -- A generated TOKENLIST column is the Spanner equivalent of Postgres' tsvector.
  -- HIDDEN keeps it out of SELECT *.
  embed_tokens TOKENLIST AS (TOKENIZE_FULLTEXT(embed_input)) HIDDEN,
) PRIMARY KEY (doc_id, chunk_index),
  INTERLEAVE IN PARENT Documents ON DELETE CASCADE""",

# SEARCH() only works when a search index exists over the TOKENLIST column being searched.
"""CREATE SEARCH INDEX IF NOT EXISTS ChunksSearchIndex
   ON Chunks(embed_tokens)
   OPTIONS (sort_order_sharding = true)""",

# chunk_id is our stable handle; a unique index lets us look up and reference by it.
"""CREATE UNIQUE INDEX IF NOT EXISTS ChunksByChunkId ON Chunks(chunk_id)""",

"""CREATE TABLE IF NOT EXISTS GraphNode (
  node_id     STRING(128) NOT NULL,
  node_type   STRING(64)  NOT NULL,   -- supplier | bid | contract | certification | ...
  label       STRING(MAX) NOT NULL,   -- human-readable name
  degree      INT64,                  -- cached, used by the hub guard during expansion
  properties  JSON,
  source      STRING(32),             -- 'metadata' | 'relational'
) PRIMARY KEY (node_id)""",

"""CREATE TABLE IF NOT EXISTS GraphEdge (
  src_id     STRING(128) NOT NULL,
  dst_id     STRING(128) NOT NULL,
  predicate  STRING(128) NOT NULL,
  is_reverse BOOL NOT NULL,           -- TRUE = the mirrored copy, for undirected traversal
  source     STRING(32),
  properties JSON,
  CONSTRAINT FK_Edge_Src FOREIGN KEY (src_id) REFERENCES GraphNode (node_id),
  CONSTRAINT FK_Edge_Dst FOREIGN KEY (dst_id) REFERENCES GraphNode (node_id),
) PRIMARY KEY (src_id, dst_id, predicate)""",

"""CREATE INDEX IF NOT EXISTS GraphEdgeByDst ON GraphEdge(dst_id, src_id)""",

"""CREATE TABLE IF NOT EXISTS NodeMention (
  node_id   STRING(128) NOT NULL,
  chunk_id  STRING(160) NOT NULL,
  doc_id    STRING(128) NOT NULL,
  method    STRING(32),               -- metadata | id_token | label
) PRIMARY KEY (node_id, chunk_id)""",

"""CREATE INDEX IF NOT EXISTS NodeMentionByChunk ON NodeMention(chunk_id)""",
]

run_ddl(DDL)
print("Tables ready: Documents, Chunks, GraphNode, GraphEdge, NodeMention (+ search index).")

### 4b — The property graph

This is the Spanner Graph declaration. It creates **no new storage** — it is a view over
`GraphNode` and `GraphEdge` that teaches Spanner how to read them as a graph.

`SOURCE KEY` / `DESTINATION KEY` are what turn a row in `GraphEdge` into an actual edge. There
is no `ALTER PROPERTY GRAPH`; you re-issue `CREATE OR REPLACE` to change it.

In [ ]:
PROPERTY_GRAPH_DDL = """
CREATE OR REPLACE PROPERTY GRAPH {GRAPH}
  NODE TABLES (
    GraphNode
      KEY (node_id)
      LABEL Entity
        PROPERTIES (node_id, node_type, label, degree)
  )
  EDGE TABLES (
    GraphEdge
      KEY (src_id, dst_id, predicate)
      SOURCE      KEY (src_id) REFERENCES GraphNode (node_id)
      DESTINATION KEY (dst_id) REFERENCES GraphNode (node_id)
      LABEL Rel
        PROPERTIES (predicate, is_reverse, source)
  )
""".replace("{GRAPH}", GRAPH_NAME)

run_ddl([PROPERTY_GRAPH_DDL])
print(f"Property graph '{GRAPH_NAME}' created over GraphNode / GraphEdge.")

# ---- Optional: an ANN vector index. Off by default (USE_VECTOR_INDEX in cell 2). ----
# Exact KNN with COSINE_DISTANCE is fast below ~100k chunks and has NO query restrictions.
# An ANN index is faster above that, but the query must then satisfy all of:
#   * use APPROX_COSINE_DISTANCE (matching the index's distance_type)
#   * that call must be the SOLE sort key in ORDER BY, followed by LIMIT
#   * filter out unindexed rows explicitly: WHERE embedding IS NOT NULL
#   * ideally pin it with @{force_index=...} — selection is not automatic
if USE_VECTOR_INDEX:
    run_ddl([f"""CREATE VECTOR INDEX IF NOT EXISTS ChunksEmbeddingIndex
                 ON Chunks(embedding)
                 WHERE embedding IS NOT NULL
                 OPTIONS (distance_type = 'COSINE', tree_depth = 2, num_leaves = 1000)"""])
    print("Vector index created — remember the ANN query rules above.")
else:
    print("Vector index skipped (exact KNN). Set USE_VECTOR_INDEX=True above ~100k chunks.")

## 5 — Embeddings

Generated in Python and inserted as plain arrays. Spanner *can* call Vertex AI itself via
`CREATE MODEL` + `ML.PREDICT`, which is genuinely useful for embedding the **query** inside the
same statement — there's a commented example at the bottom of this cell. For bulk ingestion,
Python wins: better batching, retry control, and visible cost.

In [ ]:
import tiktoken
from google import genai
from google.genai.types import EmbedContentConfig, GenerateContentConfig

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
_enc = tiktoken.get_encoding("cl100k_base")

def n_tokens(t): return len(_enc.encode(t))
def _truncate(t, mx=MAX_EMBED_TOKENS):
    tk = _enc.encode(t)
    return t if len(tk) <= mx else _enc.decode(tk[:mx])

def _l2(v):
    """Cosine distance only cares about direction, but normalising keeps distances
    comparable and well-behaved after dimensionality truncation."""
    n = math.sqrt(sum(x * x for x in v))
    return [x / n for x in v] if n else v

_TRANSIENT = ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500", "INTERNAL", "DEADLINE")
def _with_retry(fn, attempts=5, base=1.0):
    for i in range(attempts):
        try:
            return fn()
        except Exception as e:
            if i == attempts - 1 or not any(s in str(e) for s in _TRANSIENT):
                raise
            time.sleep(base * (2 ** i) + random.random())

def embed_texts(texts, task_type, batch_size=16):
    """task_type must be RETRIEVAL_DOCUMENT for stored chunks and RETRIEVAL_QUERY for
    questions — the model encodes the two roles differently and mixing them silently
    costs accuracy."""
    cfg = EmbedContentConfig(task_type=task_type, output_dimensionality=EMBED_DIM)
    out = []
    for i in range(0, len(texts), batch_size):
        batch = [_truncate(t) for t in texts[i:i + batch_size]]
        try:
            r = _with_retry(lambda: genai_client.models.embed_content(
                model=EMBED_MODEL, contents=batch, config=cfg))
            out.extend(e.values for e in r.embeddings)
        except Exception:
            for t in batch:      # some quota configurations reject batches
                r = _with_retry(lambda t=t: genai_client.models.embed_content(
                    model=EMBED_MODEL, contents=[t], config=cfg))
                out.append(r.embeddings[0].values)
    return [_l2(list(v)) for v in out]

def embed_query(q):
    return embed_texts([q], "RETRIEVAL_QUERY")[0]

_p = embed_query("hello")
assert len(_p) == EMBED_DIM, f"expected {EMBED_DIM}, got {len(_p)}"
print(f"Vertex OK — {EMBED_MODEL} → {len(_p)} dims")

# ---- Optional: let Spanner embed the query itself, removing a round trip from the read path.
# run_ddl([f"""CREATE MODEL IF NOT EXISTS EmbeddingsModel
#   INPUT(content STRING(MAX), task_type STRING(MAX))
#   OUTPUT(embeddings STRUCT<values ARRAY<FLOAT32>>)
#   REMOTE OPTIONS (endpoint =
#     '//aiplatform.googleapis.com/projects/{PROJECT_ID}/locations/{LOCATION}'
#     '/publishers/google/models/{EMBED_MODEL}')"""])
# Then inside a query:
#   WITH q AS (SELECT embeddings.values AS v FROM ML.PREDICT(MODEL EmbeddingsModel,
#              (SELECT @question AS content, "RETRIEVAL_QUERY" AS task_type),
#              STRUCT(1536 AS outputDimensionality)))

## 6 — Chunking

Identical to `1.ipynb`: split on paragraphs, then sentences, then tokens as a last resort;
carry whole sentences as overlap. Each chunk is stored twice — `content` (clean, shown to you)
and `embed_input` (a contextual header + the text, which is what gets embedded *and* tokenized).

That header is why a chunk reading *"Customer satisfaction: 4.8/5"* is findable at all: on its
own it has no supplier attached.

In [ ]:
_PARA = re.compile(r"\n\s*\n")
_SENT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9(\"'])")

def _hard_split(unit, mx):
    tk = _enc.encode(unit)
    return [_enc.decode(tk[i:i + mx]) for i in range(0, len(tk), mx)]

def split_text(text, max_tokens=CHUNK_TOKENS, overlap_tokens=CHUNK_OVERLAP_TOKENS):
    units = []
    for para in _PARA.split(text.strip()):
        para = para.strip()
        if not para:
            continue
        for sent in _SENT.split(para):
            sent = sent.strip()
            if sent:
                units.extend(_hard_split(sent, max_tokens)
                             if n_tokens(sent) > max_tokens else [sent])
    chunks, buf, buf_tokens = [], [], 0
    for u in units:
        t = n_tokens(u)
        if buf and buf_tokens + t > max_tokens:
            chunks.append(" ".join(buf))
            keep, kept = [], 0
            for prev in reversed(buf):          # overlap in WHOLE sentences
                pt = n_tokens(prev)
                if kept + pt > overlap_tokens:
                    break
                keep.insert(0, prev); kept += pt
            buf, buf_tokens = keep, kept
        buf.append(u); buf_tokens += t
    if buf:
        chunks.append(" ".join(buf))
    return chunks

HEADER_FIELDS = ["supplier_name", "category", "document_type", "bid_date", "status"]

def contextual_header(doc_id, title, metadata):
    bits = [f"Document: {title}", f"ID: {doc_id}"]
    bits += [f"{f.replace('_', ' ').title()}: {metadata[f]}"
             for f in HEADER_FIELDS if metadata.get(f)]
    return " | ".join(bits)

def build_chunks(doc_id, title, content, metadata):
    header = contextual_header(doc_id, title, metadata)
    return [{"chunk_index": i, "content": piece,
             "embed_input": f"{header}\n\n{piece}", "token_count": n_tokens(piece)}
            for i, piece in enumerate(split_text(content))]

print("Chunker ready.")

## 7 — The corpus

The same 16 procurement documents as `1.ipynb`, so you can compare the two engines on
identical data. Every entity carries a stable id reused across documents, and each document's
metadata declares the `entities` it touches and the `relations` between them — that is what
cell 9 turns into the graph, with no LLM extraction.

In [ ]:
# ============================================================================================
# SEED CORPUS — 15 food-service procurement documents for Riverside Unified School District.
#
# These are written to be RAG-realistic AND graph-ready:
#   • Every entity carries a stable ID (SUP-1042, RFP-2024-0412, BID-2024-089, CTR-…, PER-…)
#     and that SAME id is used in every document that mentions it.
#   • Facts are consistent across documents on purpose — a price quoted in a bid reappears in
#     the award memo and the contract, so cross-document questions have one right answer.
#   • metadata["entities"] lists every id the document touches; metadata["relations"] states
#     the relationships as (subject, predicate, object) triples. Nothing reads them yet — they
#     are there so a knowledge graph can be built later without re-parsing the prose.
#   • Exact tokens are sprinkled throughout (SKUs, "Net-45", "4.8/5", "$3.20/case", percentages)
#     because those are precisely where dense vectors are weak and the keyword leg earns its keep.
#
# The world, in one paragraph: Riverside USD (ORG-RUSD) ran three RFPs in spring 2024. Produce
# went to Valle Verde, dairy to Globex, frozen/protein split between Tidewater and Northlake.
# Tidewater owns Harbor Point Seafood, is the weakest performer, and has a certificate expiring.
# Acme Foods bid lower than Valle Verde on produce and still lost — on food safety and delivery.
# ============================================================================================

SOURCE_DOCS = {

# ─────────────────────────────────── RFPs ───────────────────────────────────
"rfp-2024-0412-produce": {
    "title": "RFP-2024-0412 — Fresh Produce Supply, Riverside USD, SY 2024-25",
    "content": (
        "REQUEST FOR PROPOSAL RFP-2024-0412\n"
        "Fresh Produce Supply for School Year 2024-25\n"
        "Issued by: Riverside Unified School District (ORG-RUSD), Nutrition Services Department\n"
        "Issue date: 2024-03-18. Original proposal due date: 2024-04-26, 2:00 PM PT. "
        "Extended to 2024-05-03 by Addendum No. 1.\n\n"

        "1. PURPOSE AND SCOPE\n"
        "Riverside Unified School District solicits proposals from qualified distributors for the "
        "supply and delivery of fresh fruits and vegetables to 42 school sites serving approximately "
        "28,400 students and 3.1 million meals annually. The estimated annual contract value is "
        "1,850,000 USD. The contract term runs 2024-08-01 through 2025-07-31, with two optional "
        "one-year renewals exercisable at the District's sole discretion.\n\n"

        "2. AWARD STRUCTURE\n"
        "The District intends to award a single prime contract for all produce categories. Partial "
        "or lot-based awards will be considered only if no single proposer can demonstrate capacity "
        "for the full scope. Cedar Valley Unified School District (ORG-CVUSD), 19 sites and 11,200 "
        "students, may piggyback on the resulting contract through the Central Valley Purchasing "
        "Cooperative under identical pricing and terms.\n\n"

        "3. EVALUATION CRITERIA\n"
        "Proposals are scored out of 100 points on the following weightings:\n"
        "  Price and total cost of ownership .......... 40%\n"
        "  Quality assurance and food safety .......... 25%\n"
        "  Delivery capability and reliability ........ 20%\n"
        "  Local sourcing commitment .................. 10%\n"
        "  MWBE participation ......................... 5%\n"
        "Price is scored on the extended annual basket total, not on unit price alone. A proposer "
        "may not be awarded on price advantage if it scores below 15 of 25 available points on "
        "quality assurance and food safety.\n\n"

        "4. MANDATORY REQUIREMENTS\n"
        "Proposers must hold a current GlobalG.A.P. or equivalent Good Agricultural Practices "
        "certification, and a current SQF Level 2 or higher facility certification for each "
        "distribution centre servicing this contract. Proposers must carry commercial general "
        "liability insurance of not less than 2,000,000 USD per occurrence and product liability "
        "of not less than 5,000,000 USD aggregate. A HACCP plan covering receiving, cold storage "
        "and transport must be submitted with the proposal.\n\n"

        "5. DELIVERY REQUIREMENTS\n"
        "Deliveries are required a minimum of twice weekly to each site. The original delivery "
        "window of 5:00 AM to 9:00 AM was revised to 4:30 AM to 8:30 AM by Addendum No. 1. "
        "Product must arrive at 34-41 degrees Fahrenheit for temperature-controlled items. Any "
        "delivery arriving outside the window without 24 hours prior notice is recorded as a late "
        "delivery for the purposes of the on-time performance metric.\n\n"

        "6. PERFORMANCE STANDARDS\n"
        "The awarded supplier must maintain a fill rate of at least 97% measured monthly by line "
        "item, and an on-time delivery rate of at least 95% measured monthly by delivery event. "
        "Quality rejection rate must remain below 2.0% of cases delivered.\n\n"

        "7. LOCAL SOURCING\n"
        "Points under the local sourcing criterion are awarded on the percentage of annual spend "
        "sourced from growers located within 250 miles of the District's central kitchen in "
        "Riverside, California. Addendum No. 1 clarified that the 250-mile radius is measured from "
        "the grower's primary production site, not from the distributor's warehouse.\n\n"

        "8. SUBMISSION\n"
        "Direct all questions to Miguel Arredondo (PER-MARREDONDO), Procurement Manager, "
        "purchasing@riversideusd.example.org. Questions closed 2024-04-12. Responses were issued "
        "in Addendum No. 1 on 2024-04-19. Six suppliers were formally solicited: Valle Verde "
        "Produce Co., Acme Foods Distribution, Globex Dairy Cooperative, Tidewater Frozen Holdings, "
        "Northlake Protein Partners and Summit Beverage Distributors."
    ),
    "metadata": {
        "document_type": "rfp", "category": "Produce",
        "rfp_id": "RFP-2024-0412", "buyer_id": "ORG-RUSD",
        "buyer_name": "Riverside Unified School District",
        "issue_date": "2024-03-18", "due_date": "2024-05-03",
        "estimated_value_usd": 1850000, "status": "awarded",
        "entities": ["RFP-2024-0412", "ORG-RUSD", "ORG-CVUSD", "PER-MARREDONDO",
                     "SUP-1042", "SUP-1077", "SUP-1103", "SUP-1156", "SUP-1189", "SUP-1204"],
        "relations": [
            ["ORG-RUSD", "ISSUED", "RFP-2024-0412"],
            ["RFP-2024-0412", "SOLICITS", "SUP-1042"],
            ["RFP-2024-0412", "SOLICITS", "SUP-1077"],
            ["RFP-2024-0412", "COVERS_CATEGORY", "Produce"],
            ["RFP-2024-0412", "AMENDED_BY", "ADD-2024-0412-001"],
            ["ORG-CVUSD", "MAY_PIGGYBACK_ON", "RFP-2024-0412"],
            ["PER-MARREDONDO", "CONTACT_FOR", "RFP-2024-0412"],
            ["PER-MARREDONDO", "WORKS_FOR", "ORG-RUSD"],
        ],
    },
},

"rfp-2024-0455-dairy": {
    "title": "RFP-2024-0455 — Dairy and Refrigerated Products, Riverside USD, SY 2024-25",
    "content": (
        "REQUEST FOR PROPOSAL RFP-2024-0455\n"
        "Dairy and Refrigerated Products for School Year 2024-25\n"
        "Issued by: Riverside Unified School District (ORG-RUSD)\n"
        "Issue date: 2024-04-02. Proposal due date: 2024-05-10, 2:00 PM PT.\n\n"

        "1. SCOPE\n"
        "The District seeks a single supplier for fluid milk, cultured dairy, cheese and "
        "refrigerated ready-to-eat items across 42 sites. Estimated annual value is 2,400,000 USD. "
        "Fluid milk alone represents approximately 4.6 million half-pint units annually. The "
        "contract term is 2024-08-01 through 2025-07-31 with two optional one-year renewals.\n\n"

        "2. EVALUATION CRITERIA\n"
        "  Price ........................................ 45%\n"
        "  Cold chain integrity and food safety ......... 25%\n"
        "  Delivery capability .......................... 20%\n"
        "  Sustainability and packaging ................. 10%\n\n"

        "3. COLD CHAIN REQUIREMENTS\n"
        "This is the District's controlling technical requirement. All fluid dairy must be "
        "maintained at 34 to 38 degrees Fahrenheit continuously from the processing plant to the "
        "point of delivery. Proposers must describe their telemetry capability: the District "
        "requires continuous temperature logging with data retained for a minimum of 24 months and "
        "made available to the District on request within two business days. Any load arriving "
        "above 41 degrees Fahrenheit is rejected in full at the supplier's cost and is recorded as "
        "a quality rejection event.\n\n"

        "4. MANDATORY REQUIREMENTS\n"
        "Proposers must hold a current SQF Level 2 or higher certification and a valid California "
        "Grade A dairy permit. Proposers must demonstrate the ability to deliver to all 42 sites "
        "within a four-hour window and to provide emergency replacement delivery within six hours "
        "of a rejected load.\n\n"

        "5. PACKAGING AND SUSTAINABILITY\n"
        "The District's Board resolution 2023-14 commits to reducing single-use plastic in "
        "nutrition services by 30% by 2027. Proposals are scored on paper-based carton use, "
        "recyclability, and any crate return or reuse programme offered at no additional charge.\n\n"

        "6. PRICE ADJUSTMENT\n"
        "Because fluid milk pricing is tied to federal milk marketing order Class I mover pricing, "
        "the District will accept monthly price adjustment on fluid milk line items only, indexed "
        "to the announced Class I mover, provided the proposer discloses its fixed markup over the "
        "mover at proposal time and that markup remains fixed for the contract term. All non-fluid "
        "line items are firm-fixed for the contract year, adjustable at renewal only, capped at "
        "CPI-U food-at-home or 4%, whichever is lower.\n\n"

        "7. PERFORMANCE STANDARDS\n"
        "Fill rate of at least 98% by line item measured monthly, given that a missed milk delivery "
        "cannot be absorbed by menu substitution. On-time delivery of at least 96%. Quality "
        "rejection rate below 1.0%.\n\n"

        "8. SUBMISSION\n"
        "Questions to Miguel Arredondo (PER-MARREDONDO), Procurement Manager. Three suppliers were "
        "solicited: Globex Dairy Cooperative (SUP-1103), Acme Foods Distribution (SUP-1077) and "
        "Tidewater Frozen Holdings (SUP-1156). One responsive proposal was received."
    ),
    "metadata": {
        "document_type": "rfp", "category": "Dairy",
        "rfp_id": "RFP-2024-0455", "buyer_id": "ORG-RUSD",
        "buyer_name": "Riverside Unified School District",
        "issue_date": "2024-04-02", "due_date": "2024-05-10",
        "estimated_value_usd": 2400000, "status": "awarded",
        "entities": ["RFP-2024-0455", "ORG-RUSD", "PER-MARREDONDO",
                     "SUP-1103", "SUP-1077", "SUP-1156"],
        "relations": [
            ["ORG-RUSD", "ISSUED", "RFP-2024-0455"],
            ["RFP-2024-0455", "SOLICITS", "SUP-1103"],
            ["RFP-2024-0455", "SOLICITS", "SUP-1077"],
            ["RFP-2024-0455", "SOLICITS", "SUP-1156"],
            ["RFP-2024-0455", "COVERS_CATEGORY", "Dairy"],
        ],
    },
},

"rfp-2024-0489-frozen-protein": {
    "title": "RFP-2024-0489 — Frozen Foods, Protein and Seafood, Riverside USD, SY 2024-25",
    "content": (
        "REQUEST FOR PROPOSAL RFP-2024-0489\n"
        "Frozen Foods, Protein and Seafood for School Year 2024-25\n"
        "Issued by: Riverside Unified School District (ORG-RUSD)\n"
        "Issue date: 2024-04-22. Proposal due date: 2024-05-31, 2:00 PM PT.\n\n"

        "1. SCOPE AND LOT STRUCTURE\n"
        "Estimated annual value 3,200,000 USD, the District's largest single solicitation for "
        "SY 2024-25. Unlike RFP-2024-0412 and RFP-2024-0455, this solicitation is structured in "
        "three lots and proposers may bid on any or all:\n"
        "  Lot 1 (LOT-0489-1) — Frozen prepared foods, frozen vegetables, frozen bakery. Est. 1,450,000 USD.\n"
        "  Lot 2 (LOT-0489-2) — Frozen and further-processed protein, including seafood. Est. 1,180,000 USD.\n"
        "  Lot 3 (LOT-0489-3) — Fresh and frozen poultry. Est. 570,000 USD.\n"
        "The District reserves the right to award each lot to a different supplier where doing so "
        "produces the best value.\n\n"

        "2. EVALUATION CRITERIA\n"
        "  Price ........................................ 35%\n"
        "  Food safety and traceability ................. 30%\n"
        "  Delivery capability .......................... 20%\n"
        "  Product range and menu fit ................... 15%\n"
        "Food safety carries a heavier weight here than in the District's other solicitations "
        "because of the protein and seafood content.\n\n"

        "3. FOOD SAFETY AND TRACEABILITY\n"
        "Proposers must hold SQF Level 2 or higher for every facility touching product under this "
        "contract, and must operate under a documented HACCP plan. Seafood items require Marine "
        "Stewardship Council Chain of Custody certification or an equivalent recognised by the "
        "District's Nutrition Services Director. Proposers must be able to trace any delivered lot "
        "to its production facility and production date within four hours of a District request. "
        "Where a proposer intends to fulfil any portion of Lot 2 through a subsidiary or affiliated "
        "entity, that entity must be named in the proposal and must independently satisfy every "
        "food safety requirement in this section.\n\n"

        "4. USDA FOODS PROCESSING\n"
        "The District diverts USDA Foods entitlement poultry and beef to further processing. "
        "Proposers must describe their capability to accept diverted commodity, track entitlement "
        "value, and reflect the commodity offset on invoices as a separate line.\n\n"

        "5. DELIVERY REQUIREMENTS\n"
        "Frozen product must arrive at 0 degrees Fahrenheit or below. Fresh poultry must arrive at "
        "28 to 34 degrees Fahrenheit. Deliveries are required weekly for frozen and twice weekly "
        "for fresh poultry. The District's central kitchen has 11 dock positions and will schedule "
        "arrival windows in 45-minute slots.\n\n"

        "6. PERFORMANCE STANDARDS\n"
        "Fill rate of at least 96% by line item, on-time delivery of at least 95%, quality "
        "rejection rate below 1.5%. A supplier falling below 92% on-time in any two consecutive "
        "months is subject to a mandatory corrective action plan.\n\n"

        "7. SUBMISSION\n"
        "Questions to Miguel Arredondo (PER-MARREDONDO). Four suppliers were solicited: Tidewater "
        "Frozen Holdings (SUP-1156), Northlake Protein Partners (SUP-1189), Acme Foods Distribution "
        "(SUP-1077) and Harbor Point Seafood (SUP-1157). Harbor Point Seafood did not submit an "
        "independent proposal and was instead named as the seafood fulfilment affiliate within the "
        "Tidewater Frozen Holdings proposal."
    ),
    "metadata": {
        "document_type": "rfp", "category": "Frozen Foods",
        "rfp_id": "RFP-2024-0489", "buyer_id": "ORG-RUSD",
        "buyer_name": "Riverside Unified School District",
        "issue_date": "2024-04-22", "due_date": "2024-05-31",
        "estimated_value_usd": 3200000, "status": "awarded",
        "entities": ["RFP-2024-0489", "ORG-RUSD", "PER-MARREDONDO",
                     "SUP-1156", "SUP-1189", "SUP-1077", "SUP-1157"],
        "relations": [
            ["ORG-RUSD", "ISSUED", "RFP-2024-0489"],
            ["RFP-2024-0489", "SOLICITS", "SUP-1156"],
            ["RFP-2024-0489", "SOLICITS", "SUP-1189"],
            ["RFP-2024-0489", "SOLICITS", "SUP-1077"],
            ["RFP-2024-0489", "SOLICITS", "SUP-1157"],
            ["RFP-2024-0489", "HAS_LOT", "LOT-0489-1"],
            ["RFP-2024-0489", "HAS_LOT", "LOT-0489-2"],
            ["RFP-2024-0489", "HAS_LOT", "LOT-0489-3"],
            ["SUP-1157", "SUBSIDIARY_OF", "SUP-1156"],
        ],
    },
},

"addendum-rfp-0412-001": {
    "title": "Addendum No. 1 to RFP-2024-0412 — Fresh Produce Supply",
    "content": (
        "ADDENDUM NO. 1 TO RFP-2024-0412 (document reference ADD-2024-0412-001)\n"
        "Issued: 2024-04-19 by Riverside Unified School District (ORG-RUSD)\n"
        "Acknowledgement of this addendum is mandatory. A proposal that fails to acknowledge "
        "Addendum No. 1 will be deemed non-responsive.\n\n"

        "PART A — CHANGES TO THE SOLICITATION\n\n"
        "A.1 Proposal due date. The due date is extended from 2024-04-26, 2:00 PM PT to "
        "2024-05-03, 2:00 PM PT. No further extension will be granted.\n\n"
        "A.2 Delivery window. Section 5 is amended. The delivery window is changed from 5:00 AM "
        "to 9:00 AM, to 4:30 AM to 8:30 AM. This change was made at the request of two site "
        "kitchen managers whose meal service begins at 7:15 AM and who reported insufficient "
        "receiving time under the original window.\n\n"
        "A.3 Local sourcing definition. Section 7 is clarified. The 250-mile radius is measured "
        "from the grower's primary production site to the District's central kitchen in Riverside, "
        "California. It is not measured from the distributor's warehouse. Proposers claiming local "
        "sourcing percentages must be able to substantiate them with grower addresses on request.\n\n"
        "A.4 Insurance. Section 4 is amended to clarify that the 5,000,000 USD product liability "
        "requirement may be met through a combination of primary and umbrella coverage.\n\n"

        "PART B — QUESTIONS AND ANSWERS\n"
        "Questions closed 2024-04-12. Eleven questions were received from three suppliers.\n\n"
        "Q1 (Valle Verde Produce Co., SUP-1042): May a proposer offer a volume discount tied to "
        "order value rather than to annual spend?\n"
        "A1: Yes. Volume discount structures will be evaluated as part of the extended annual "
        "basket total. State the threshold and the discount percentage clearly.\n\n"
        "Q2 (Acme Foods Distribution, SUP-1077): Will the District consider a proposer whose SQF "
        "certification for one of two distribution centres is pending renewal at proposal time?\n"
        "A2: No. Section 4 is a mandatory requirement. Every distribution centre servicing this "
        "contract must hold a current certificate on the proposal due date. A certificate in "
        "renewal is not a current certificate for the purposes of this solicitation.\n\n"
        "Q3 (Valle Verde Produce Co., SUP-1042): Is the 97% fill rate measured by case or by line "
        "item?\n"
        "A3: By line item, measured monthly. A line item short-shipped in any quantity counts as "
        "unfilled for that delivery event.\n\n"
        "Q4 (Acme Foods Distribution, SUP-1077): Can the twice-weekly delivery requirement be met "
        "with a single larger delivery for smaller sites?\n"
        "A4: No. Fresh produce shelf life at site level does not support consolidation. Twice "
        "weekly is a minimum at every site regardless of site size.\n\n"
        "Q5 (Tidewater Frozen Holdings, SUP-1156): Does the District intend to award produce and "
        "frozen categories to a single supplier?\n"
        "A5: No. RFP-2024-0412 covers produce only. Frozen, protein and seafood are solicited "
        "separately under RFP-2024-0489.\n\n"
        "Q6-Q11 concerned invoice formatting, EDI capability, the District's site list, crate "
        "return, substitution approval workflow and payment timing. The District confirmed that "
        "standard payment terms are Net-30 from receipt of a correct invoice, that early payment "
        "discount offers are welcome and will be evaluated within the price criterion, and that "
        "all substitutions require written approval from the Nutrition Services Director "
        "(PER-DWHITFIELD) before delivery.\n\n"
        "All other terms and conditions of RFP-2024-0412 remain unchanged."
    ),
    "metadata": {
        "document_type": "addendum", "category": "Produce",
        "addendum_id": "ADD-2024-0412-001", "rfp_id": "RFP-2024-0412",
        "buyer_id": "ORG-RUSD", "issue_date": "2024-04-19", "status": "issued",
        "entities": ["ADD-2024-0412-001", "RFP-2024-0412", "ORG-RUSD",
                     "SUP-1042", "SUP-1077", "SUP-1156", "PER-DWHITFIELD"],
        "relations": [
            ["ADD-2024-0412-001", "AMENDS", "RFP-2024-0412"],
            ["ORG-RUSD", "ISSUED", "ADD-2024-0412-001"],
            ["SUP-1042", "ASKED_QUESTION_IN", "ADD-2024-0412-001"],
            ["SUP-1077", "ASKED_QUESTION_IN", "ADD-2024-0412-001"],
            ["SUP-1156", "ASKED_QUESTION_IN", "ADD-2024-0412-001"],
            ["PER-DWHITFIELD", "APPROVES", "substitutions"],
        ],
    },
},

# ─────────────────────────────────── BIDS ───────────────────────────────────
"bid-2024-089-valleverde": {
    "title": "BID-2024-089 — Valle Verde Produce Co. response to RFP-2024-0412",
    "content": (
        "PROPOSAL BID-2024-089\n"
        "Submitted by: Valle Verde Produce Co. (SUP-1042)\n"
        "In response to: RFP-2024-0412, Fresh Produce Supply, Riverside Unified School District\n"
        "Submitted: 2024-05-01, 10:42 AM PT. Addendum No. 1 acknowledged.\n"
        "Total extended annual basket: 1,742,880 USD\n"
        "Primary contact: Elena Marquez (PER-EMARQUEZ), VP Sales, "
        "e.marquez@valleverdeproduce.example.com\n\n"

        "1. COMPANY\n"
        "Valle Verde Produce Co. was established in 1998 and is headquartered in Salinas, "
        "California. We employ 240 people and operate a single 96,000 square foot distribution "
        "centre in Fresno, California, certified SQF Level 2 (certificate expires 2025-09-14). "
        "We hold GlobalG.A.P. certification through 2025-06-30 and a USDA Organic handler "
        "certificate through 2025-11-01. We currently serve 38 K-12 districts across central and "
        "southern California.\n\n"

        "2. PRICING\n"
        "All prices are per case, delivered, firm-fixed for the contract year:\n"
        "  Romaine lettuce, 24 ct .............. 3.20 USD/case   SKU VV-PRD-1180\n"
        "  Spinach, baby, 4x2.5 lb ............. 4.10 USD/case   SKU VV-PRD-1204\n"
        "  Carrots, whole, 50 lb ............... 2.05 USD/case   SKU VV-PRD-1311\n"
        "  Brussels sprouts, 25 lb ............. 2.85 USD/case   SKU VV-PRD-1422\n"
        "  Bell peppers, mixed colour, 25 lb ... 3.50 USD/case   SKU VV-PRD-1508\n"
        "  Roma tomatoes, 25 lb ................ 3.75 USD/case   SKU VV-PRD-1560\n"
        "  Broccoli crowns, 20 lb .............. 4.40 USD/case   SKU VV-PRD-1614\n"
        "A further 61 line items are priced in Attachment C.\n\n"

        "3. COMMERCIAL TERMS\n"
        "Payment terms: Net-30 from receipt of a correct invoice. We do not offer an early payment "
        "discount. Minimum order value: 100 USD per site per delivery. Volume discount: 5% applied "
        "at the invoice level on any single site delivery exceeding 500 USD. Price escalation is "
        "not sought during the initial contract year. At renewal we request adjustment capped at "
        "CPI-U food-at-home or 4%, whichever is lower.\n\n"

        "4. DELIVERY\n"
        "We commit to twice-weekly delivery to all 42 sites on Tuesdays and Thursdays within the "
        "revised 4:30 AM to 8:30 AM window established by Addendum No. 1. We operate 34 refrigerated "
        "vehicles and will dedicate 6 to this contract. Emergency replacement delivery is available "
        "within four hours during the school week.\n\n"

        "5. LOCAL SOURCING\n"
        "We commit that 64% of annual spend under this contract will be sourced from growers whose "
        "primary production site lies within 250 miles of the District's central kitchen, as "
        "clarified by Addendum No. 1. Named growers include Sandoval Family Farms (Oxnard, 118 "
        "miles), Mesa Grande Growers (Coachella, 74 miles) and Ridgeline Organics (Bakersfield, "
        "162 miles). We will report actual local spend quarterly.\n\n"

        "6. QUALITY ASSURANCE\n"
        "Every inbound lot is inspected against USDA grade standards at our Fresno facility. We "
        "operate a documented HACCP plan covering receiving, cold storage and transport, last "
        "audited 2024-02-08 with zero non-conformances. Product is held at 34 to 38 degrees "
        "Fahrenheit and we log trailer temperature continuously with data retained 36 months.\n\n"

        "7. MWBE\n"
        "Valle Verde Produce Co. is a certified Women's Business Enterprise, California "
        "certification WBE-CA-22841, and 31% of our contract spend flows to MWBE-certified growers "
        "and service providers."
    ),
    "metadata": {
        "document_type": "bid_response", "category": "Produce",
        "bid_id": "BID-2024-089", "rfp_id": "RFP-2024-0412",
        "supplier_id": "SUP-1042", "supplier_name": "Valle Verde Produce Co.",
        "buyer_id": "ORG-RUSD", "bid_date": "2024-05-01",
        "total_bid_amount_usd": 1742880, "payment_terms": "Net-30",
        "status": "awarded", "contract_id": "CTR-2024-0412-VV",
        "entities": ["BID-2024-089", "RFP-2024-0412", "SUP-1042", "ORG-RUSD",
                     "PER-EMARQUEZ", "CTR-2024-0412-VV",
                     "VV-PRD-1180", "VV-PRD-1204", "VV-PRD-1311", "VV-PRD-1422",
                     "VV-PRD-1508", "VV-PRD-1560", "VV-PRD-1614"],
        "relations": [
            ["SUP-1042", "SUBMITTED", "BID-2024-089"],
            ["BID-2024-089", "RESPONDS_TO", "RFP-2024-0412"],
            ["BID-2024-089", "RESULTED_IN", "CTR-2024-0412-VV"],
            ["BID-2024-089", "QUOTES_PRODUCT", "VV-PRD-1180"],
            ["BID-2024-089", "QUOTES_PRODUCT", "VV-PRD-1204"],
            ["PER-EMARQUEZ", "WORKS_FOR", "SUP-1042"],
            ["PER-EMARQUEZ", "CONTACT_FOR", "BID-2024-089"],
            ["SUP-1042", "HOLDS_CERTIFICATION", "CERT-VV-SQF2"],
            ["SUP-1042", "HOLDS_CERTIFICATION", "CERT-VV-GAP"],
            ["SUP-1042", "OPERATES_FACILITY", "FAC-VV-FRESNO"],
        ],
    },
},

"bid-2024-091-acme": {
    "title": "BID-2024-091 — Acme Foods Distribution response to RFP-2024-0412",
    "content": (
        "PROPOSAL BID-2024-091\n"
        "Submitted by: Acme Foods Distribution (SUP-1077)\n"
        "In response to: RFP-2024-0412, Fresh Produce Supply, Riverside Unified School District\n"
        "Submitted: 2024-05-02, 4:55 PM PT. Addendum No. 1 acknowledged.\n"
        "Total extended annual basket: 1,689,450 USD\n"
        "Primary contact: Ray Okafor (PER-ROKAFOR), Director of K-12 Sales, "
        "r.okafor@acmefoods.example.com\n\n"

        "1. COMPANY\n"
        "Acme Foods Distribution is a broadline distributor headquartered in Sacramento, "
        "California, founded 1974, with 610 employees. We operate two distribution centres: "
        "Sacramento (210,000 square feet, SQF Level 2, certificate expires 2025-07-22) and "
        "Riverside (88,000 square feet, SQF Level 2 certificate submitted for renewal on "
        "2024-04-10, decision pending as of the proposal date). We hold GlobalG.A.P. certification "
        "through 2026-03-15. We serve 71 K-12 districts.\n\n"

        "2. PRICING\n"
        "All prices per case, delivered, firm-fixed for the contract year:\n"
        "  Romaine lettuce, 24 ct .............. 3.05 USD/case   SKU AF-P-4410\n"
        "  Spinach, baby, 4x2.5 lb ............. 3.95 USD/case   SKU AF-P-4418\n"
        "  Carrots, whole, 50 lb ............... 1.98 USD/case   SKU AF-P-4433\n"
        "  Brussels sprouts, 25 lb ............. 2.90 USD/case   SKU AF-P-4460\n"
        "  Bell peppers, mixed colour, 25 lb ... 3.42 USD/case   SKU AF-P-4471\n"
        "  Roma tomatoes, 25 lb ................ 3.60 USD/case   SKU AF-P-4488\n"
        "  Broccoli crowns, 20 lb .............. 4.55 USD/case   SKU AF-P-4501\n"
        "Our extended basket total of 1,689,450 USD is 53,430 USD below the next lowest proposal, "
        "a 3.1% saving to the District.\n\n"

        "3. COMMERCIAL TERMS\n"
        "Payment terms: Net-30. Early payment discount: 1% if paid within 15 days. Minimum order "
        "value: 150 USD per site per delivery. Volume discount: 4% on any single site delivery "
        "exceeding 600 USD.\n\n"

        "4. DELIVERY\n"
        "We commit to twice-weekly delivery on Mondays and Wednesdays. We note that our Riverside "
        "distribution centre currently services 71 districts from 22 vehicles and that the revised "
        "4:30 AM window in Addendum No. 1 requires a shift start change we are working to "
        "implement. We propose a 60-day transition period during which deliveries to 9 outlying "
        "sites would arrive between 8:30 AM and 9:30 AM, outside the specified window.\n\n"

        "5. LOCAL SOURCING\n"
        "We commit that 41% of annual spend will be sourced within 250 miles of the District's "
        "central kitchen. We source through 14 grower partners and can provide addresses on "
        "request.\n\n"

        "6. QUALITY ASSURANCE\n"
        "Inbound lots are inspected against USDA grade standards. Our HACCP plan was last audited "
        "2023-11-14. That audit recorded three minor non-conformances relating to cold storage "
        "door seals and receiving log completeness at the Riverside facility; two were closed "
        "2024-01-30 and one remains open pending a capital repair scheduled for July 2024. Trailer "
        "temperature is logged at 15-minute intervals with data retained 18 months.\n\n"

        "7. MWBE\n"
        "Acme Foods Distribution is not an MWBE-certified entity. 12% of contract spend is directed "
        "to MWBE-certified suppliers and service providers."
    ),
    "metadata": {
        "document_type": "bid_response", "category": "Produce",
        "bid_id": "BID-2024-091", "rfp_id": "RFP-2024-0412",
        "supplier_id": "SUP-1077", "supplier_name": "Acme Foods Distribution",
        "buyer_id": "ORG-RUSD", "bid_date": "2024-05-02",
        "total_bid_amount_usd": 1689450, "payment_terms": "Net-30",
        "status": "not_awarded",
        "entities": ["BID-2024-091", "RFP-2024-0412", "SUP-1077", "ORG-RUSD",
                     "PER-ROKAFOR", "AF-P-4410", "AF-P-4418", "AF-P-4433",
                     "FAC-AF-SAC", "FAC-AF-RIV"],
        "relations": [
            ["SUP-1077", "SUBMITTED", "BID-2024-091"],
            ["BID-2024-091", "RESPONDS_TO", "RFP-2024-0412"],
            ["BID-2024-091", "COMPETES_WITH", "BID-2024-089"],
            ["BID-2024-091", "QUOTES_PRODUCT", "AF-P-4410"],
            ["PER-ROKAFOR", "WORKS_FOR", "SUP-1077"],
            ["SUP-1077", "OPERATES_FACILITY", "FAC-AF-SAC"],
            ["SUP-1077", "OPERATES_FACILITY", "FAC-AF-RIV"],
        ],
    },
},

"bid-2024-104-globex": {
    "title": "BID-2024-104 — Globex Dairy Cooperative response to RFP-2024-0455",
    "content": (
        "PROPOSAL BID-2024-104\n"
        "Submitted by: Globex Dairy Cooperative (SUP-1103)\n"
        "In response to: RFP-2024-0455, Dairy and Refrigerated Products, Riverside USD\n"
        "Submitted: 2024-05-08, 11:20 AM PT\n"
        "Total extended annual basket: 2,311,600 USD\n"
        "Primary contact: Priya Raghunathan (PER-PRAGHUNATHAN), Director of Institutional Sales, "
        "p.raghunathan@globexdairy.example.coop\n\n"

        "1. COMPANY\n"
        "Globex Dairy Cooperative is a member-owned cooperative of 61 family dairy farms across "
        "the San Joaquin Valley, formed in 1961. We operate one processing plant in Visalia, "
        "California, certified SQF Level 3 (certificate expires 2026-02-28), and hold California "
        "Grade A dairy permit CA-DP-4471. We supply 44 school districts and 12 hospital systems.\n\n"

        "2. PRICING\n"
        "  Milk, 1% white, half-pint carton ......... 0.2450 USD/unit   SKU GX-DRY-0101\n"
        "  Milk, fat-free chocolate, half-pint ...... 0.2675 USD/unit   SKU GX-DRY-0104\n"
        "  Milk, whole, half-pint ................... 0.2610 USD/unit   SKU GX-DRY-0108\n"
        "  String cheese, part-skim, 1 oz ........... 0.1890 USD/unit   SKU GX-DRY-0240\n"
        "  Yogurt, low-fat strawberry, 4 oz cup ..... 0.3120 USD/unit   SKU GX-DRY-0312\n"
        "  Mozzarella, shredded, 5 lb bag ........... 11.40 USD/bag     SKU GX-DRY-0455\n"
        "  Cottage cheese, 5 lb tub ................. 9.75 USD/tub      SKU GX-DRY-0470\n"
        "Fluid milk line items are quoted as the announced Class I mover plus a fixed markup of "
        "0.0385 USD per half-pint. That markup is fixed for the contract term as required by "
        "Section 6 of the solicitation. All non-fluid items are firm-fixed for the contract year.\n\n"

        "3. COMMERCIAL TERMS\n"
        "Payment terms: Net-45 from receipt of a correct invoice. Early payment discount: 2% if "
        "paid within 10 days. Minimum order value: 200 USD per site per delivery. No volume "
        "discount is offered; cooperative pricing is already at member cost plus operating margin.\n\n"

        "4. COLD CHAIN\n"
        "This is the core of our proposal. Product is held at 34 to 38 degrees Fahrenheit "
        "continuously from the Visalia plant to the point of delivery. Every trailer carries two "
        "independent telemetry units logging at 5-minute intervals, with data transmitted in "
        "real time and retained for 36 months, exceeding the 24-month requirement. The District "
        "will be issued read-only portal credentials so temperature history for any delivery can "
        "be pulled without contacting us. We guarantee cold chain integrity: any load arriving "
        "above 41 degrees Fahrenheit is replaced in full at our cost within six hours, and we do "
        "not invoice the rejected load.\n\n"

        "5. DELIVERY\n"
        "Delivery three times weekly to all 42 sites on Mondays, Wednesdays and Fridays, within a "
        "four-hour window. We operate 28 refrigerated vehicles from Visalia and will dedicate 9 to "
        "this contract. Emergency replacement within six hours as required.\n\n"

        "6. SUSTAINABILITY AND PACKAGING\n"
        "All half-pint fluid milk is supplied in paper-based gable-top cartons, recyclable in "
        "Riverside County's programme. We operate a crate return programme at no charge: crates "
        "are collected on the following scheduled delivery, which removes approximately 51 tons of "
        "plastic from the District's waste stream annually. This directly supports Board "
        "resolution 2023-14.\n\n"

        "7. QUALITY\n"
        "Our HACCP plan covers receiving, pasteurisation, packaging, cold storage and transport, "
        "last audited 2024-03-05 with zero non-conformances. We conduct finished-product microbial "
        "testing on every production lot and retain samples for 21 days past code date."
    ),
    "metadata": {
        "document_type": "bid_response", "category": "Dairy",
        "bid_id": "BID-2024-104", "rfp_id": "RFP-2024-0455",
        "supplier_id": "SUP-1103", "supplier_name": "Globex Dairy Cooperative",
        "buyer_id": "ORG-RUSD", "bid_date": "2024-05-08",
        "total_bid_amount_usd": 2311600, "payment_terms": "Net-45",
        "early_payment_discount": "2% net 10", "status": "awarded",
        "entities": ["BID-2024-104", "RFP-2024-0455", "SUP-1103", "ORG-RUSD",
                     "PER-PRAGHUNATHAN", "GX-DRY-0101", "GX-DRY-0104", "GX-DRY-0240",
                     "GX-DRY-0312", "FAC-GX-VISALIA", "CERT-GX-SQF3"],
        "relations": [
            ["SUP-1103", "SUBMITTED", "BID-2024-104"],
            ["BID-2024-104", "RESPONDS_TO", "RFP-2024-0455"],
            ["BID-2024-104", "QUOTES_PRODUCT", "GX-DRY-0101"],
            ["BID-2024-104", "QUOTES_PRODUCT", "GX-DRY-0312"],
            ["PER-PRAGHUNATHAN", "WORKS_FOR", "SUP-1103"],
            ["SUP-1103", "OPERATES_FACILITY", "FAC-GX-VISALIA"],
            ["SUP-1103", "HOLDS_CERTIFICATION", "CERT-GX-SQF3"],
        ],
    },
},

"bid-2024-117-tidewater": {
    "title": "BID-2024-117 — Tidewater Frozen Holdings response to RFP-2024-0489 (Lots 1 and 2)",
    "content": (
        "PROPOSAL BID-2024-117\n"
        "Submitted by: Tidewater Frozen Holdings (SUP-1156)\n"
        "In response to: RFP-2024-0489, Frozen Foods, Protein and Seafood, Riverside USD\n"
        "Lots bid: Lot 1 (frozen prepared, vegetables, bakery) and Lot 2 (frozen protein and "
        "seafood). We do not bid Lot 3.\n"
        "Submitted: 2024-05-29, 1:05 PM PT\n"
        "Total extended annual basket, Lots 1 and 2 combined: 2,984,220 USD\n"
        "Primary contact: Gregory Halloran (PER-GHALLORAN), SVP National Accounts, "
        "g.halloran@tidewaterfrozen.example.com\n\n"

        "1. COMPANY\n"
        "Tidewater Frozen Holdings was founded in 1987 and is headquartered in Stockton, "
        "California. We recorded 412 million USD in revenue in FY2024 and employ 1,180 people. "
        "Our categories are Frozen Foods, Dry Groceries and Meat & Poultry. We operate two "
        "distribution centres: Stockton (285,000 square feet) and Bakersfield (140,000 square "
        "feet), and a fleet of 96 refrigerated vehicles. We serve 214 K-12 districts.\n\n"

        "2. SEAFOOD FULFILMENT AFFILIATE\n"
        "As required by Section 3 of the solicitation, we name Harbor Point Seafood (SUP-1157) as "
        "the affiliate fulfilling all seafood line items under Lot 2. Harbor Point Seafood has "
        "been a wholly owned subsidiary of Tidewater Frozen Holdings since our acquisition of the "
        "business on 2021-06-30 for 34 million USD. Harbor Point holds Marine Stewardship Council "
        "Chain of Custody certification MSC-C-58817, valid through 2026-01-31, and operates under "
        "an independent seafood HACCP plan.\n\n"

        "3. PRICING, SELECTED LINE ITEMS\n"
        "  Beef patty, 100% beef, 2.0 oz, CN ....... 0.4180 USD/unit   SKU TW-PRO-2210\n"
        "  Chicken nuggets, WG breaded, 5 ct ....... 0.3925 USD/unit   SKU TW-PRO-2244\n"
        "  Pollock fillet, MSC, 3.6 oz ............. 0.8140 USD/unit   SKU HP-SEA-7702\n"
        "  Cheese pizza, WG crust, 4x6 ............. 0.6350 USD/unit   SKU TW-FRZ-3105\n"
        "  Green beans, IQF, 20 lb ................. 18.90 USD/case    SKU TW-FRZ-3320\n"
        "  Sweet corn, IQF, 20 lb .................. 16.75 USD/case    SKU TW-FRZ-3324\n"
        "  Dinner roll, WG, 1 oz ................... 0.1140 USD/unit   SKU TW-FRZ-3488\n"
        "A further 214 line items are priced in Attachment D.\n\n"

        "4. COMMERCIAL TERMS\n"
        "Payment terms: Net-45. Early payment discount: 1.5% if paid within 15 days. Minimum "
        "order value: 400 USD per delivery. Volume discount: 3% on monthly spend exceeding "
        "200,000 USD, applied as a rebate credited quarterly.\n\n"

        "5. USDA FOODS PROCESSING\n"
        "We accept diverted USDA Foods entitlement beef and poultry at both our Stockton facility "
        "and through our processor network. Entitlement value is tracked per district and appears "
        "as a separate commodity offset line on every invoice. In SY 2023-24 we processed "
        "1.9 million pounds of diverted commodity across our K-12 accounts.\n\n"

        "6. FOOD SAFETY AND TRACEABILITY\n"
        "Both distribution centres are certified SQF Level 2. The Stockton certificate expires "
        "2025-04-30 and renewal audit is scheduled for 2025-03-11. The Bakersfield certificate "
        "expires 2025-10-08. We additionally hold BRCGS AA grade at Stockton through 2025-08-19. "
        "Lot traceability is available within two hours through our warehouse management system, "
        "exceeding the four-hour requirement.\n"
        "In the interest of full disclosure, our Stockton facility's September 2024 third-party "
        "audit recorded two minor non-conformances, relating to a freezer door seal and a gap in "
        "manual temperature logging. Both were closed on 2024-10-11 and verified by the auditor.\n\n"

        "7. DELIVERY\n"
        "Weekly delivery for frozen product to all 42 sites, arriving at 0 degrees Fahrenheit or "
        "below. We will accept 45-minute scheduled dock slots at the central kitchen. We note that "
        "our current on-time performance across our K-12 book is 93.4% and we are investing in "
        "route optimisation to raise this above 95% during SY 2024-25."
    ),
    "metadata": {
        "document_type": "bid_response", "category": "Frozen Foods",
        "bid_id": "BID-2024-117", "rfp_id": "RFP-2024-0489",
        "supplier_id": "SUP-1156", "supplier_name": "Tidewater Frozen Holdings",
        "buyer_id": "ORG-RUSD", "bid_date": "2024-05-29",
        "total_bid_amount_usd": 2984220, "payment_terms": "Net-45",
        "early_payment_discount": "1.5% net 15", "status": "awarded",
        "contract_id": "CTR-2024-0489-TFH", "lots": ["LOT-0489-1", "LOT-0489-2"],
        "entities": ["BID-2024-117", "RFP-2024-0489", "SUP-1156", "SUP-1157", "ORG-RUSD",
                     "PER-GHALLORAN", "CTR-2024-0489-TFH", "TW-PRO-2210", "TW-PRO-2244",
                     "HP-SEA-7702", "TW-FRZ-3105", "FAC-TW-STOCKTON", "FAC-TW-BAKERSFIELD",
                     "CERT-TW-SQF2-STK", "CERT-HP-MSC"],
        "relations": [
            ["SUP-1156", "SUBMITTED", "BID-2024-117"],
            ["BID-2024-117", "RESPONDS_TO", "RFP-2024-0489"],
            ["BID-2024-117", "BIDS_LOT", "LOT-0489-1"],
            ["BID-2024-117", "BIDS_LOT", "LOT-0489-2"],
            ["BID-2024-117", "RESULTED_IN", "CTR-2024-0489-TFH"],
            ["SUP-1157", "SUBSIDIARY_OF", "SUP-1156"],
            ["SUP-1157", "FULFILS_CATEGORY_FOR", "BID-2024-117"],
            ["SUP-1157", "HOLDS_CERTIFICATION", "CERT-HP-MSC"],
            ["SUP-1156", "OPERATES_FACILITY", "FAC-TW-STOCKTON"],
            ["SUP-1156", "OPERATES_FACILITY", "FAC-TW-BAKERSFIELD"],
            ["PER-GHALLORAN", "WORKS_FOR", "SUP-1156"],
        ],
    },
},

"bid-2024-118-northlake": {
    "title": "BID-2024-118 — Northlake Protein Partners response to RFP-2024-0489 (Lots 2 and 3)",
    "content": (
        "PROPOSAL BID-2024-118\n"
        "Submitted by: Northlake Protein Partners (SUP-1189)\n"
        "In response to: RFP-2024-0489, Frozen Foods, Protein and Seafood, Riverside USD\n"
        "Lots bid: Lot 2 (frozen protein and seafood) and Lot 3 (fresh and frozen poultry). "
        "We do not bid Lot 1.\n"
        "Submitted: 2024-05-30, 9:15 AM PT\n"
        "Total extended annual basket, Lots 2 and 3 combined: 3,102,700 USD\n"
        "Primary contact: Sandra Kelley (PER-SKELLEY), Director of Education Sales, "
        "s.kelley@northlakeprotein.example.com\n\n"

        "1. COMPANY\n"
        "Northlake Protein Partners is a protein specialist founded in 2004, headquartered in "
        "Modesto, California, with 430 employees. We operate USDA establishment EST. 18442 in "
        "Modesto, a further-processing facility certified SQF Level 2 through 2025-12-05, and a "
        "cold storage facility in Ontario, California. We do not operate a broadline frozen "
        "business, which is why we bid Lots 2 and 3 only.\n\n"

        "2. PRICING, SELECTED LINE ITEMS\n"
        "  Chicken drumstick, fresh, bulk .......... 1.2450 USD/lb     SKU NL-PLT-5010\n"
        "  Chicken thigh, boneless, fresh .......... 2.1850 USD/lb     SKU NL-PLT-5024\n"
        "  Whole grain breaded chicken patty, 3 oz . 0.4310 USD/unit   SKU NL-PLT-5150\n"
        "  Turkey taco filling, cooked, 5 lb ....... 14.20 USD/bag     SKU NL-PRO-5402\n"
        "  Beef crumble, cooked, 5 lb .............. 17.85 USD/bag     SKU NL-PRO-5418\n"
        "  Pollock fillet, 3.6 oz, non-MSC ......... 0.7480 USD/unit   SKU NL-SEA-5900\n"
        "Our poultry pricing under Lot 3 is 6.4% below the District's SY 2023-24 contracted "
        "average and reflects direct grower relationships rather than brokered supply.\n\n"

        "3. COMMERCIAL TERMS\n"
        "Payment terms: Net-30. Early payment discount: 1% if paid within 10 days. Minimum order "
        "value: 350 USD per delivery. No volume discount offered.\n\n"

        "4. FOOD SAFETY AND TRACEABILITY\n"
        "Our Modesto facility operates under continuous USDA FSIS inspection as EST. 18442. Our "
        "HACCP plan covers receiving, further processing, freezing, cold storage and transport, "
        "last audited 2024-04-18 with one minor non-conformance concerning label verification, "
        "closed 2024-05-02. Lot traceability is available within 90 minutes.\n"
        "We note that our pollock offering under Lot 2 is not Marine Stewardship Council "
        "certified. We are able to source an MSC-certified equivalent at 0.8320 USD per unit if "
        "the District requires MSC Chain of Custody as a mandatory condition of award for seafood "
        "line items.\n\n"

        "5. DELIVERY\n"
        "Twice-weekly delivery of fresh poultry to all 42 sites at 28 to 34 degrees Fahrenheit, "
        "and weekly delivery of frozen protein at 0 degrees Fahrenheit or below. We operate 41 "
        "refrigerated vehicles and will dedicate 11 to this contract. Our current on-time "
        "performance across our education book is 95.8%.\n\n"

        "6. USDA FOODS PROCESSING\n"
        "We are an approved USDA Foods processor for beef and poultry and hold current end-product "
        "data schedules for 34 items. Entitlement drawdown is reported monthly and reflected as a "
        "commodity offset line on each invoice.\n\n"

        "7. PRODUCT RANGE\n"
        "We offer 96 line items across Lots 2 and 3. We acknowledge that this is narrower than a "
        "broadline competitor and we do not offer frozen bakery, frozen vegetables or frozen "
        "prepared entrées, which is the basis of our decision not to bid Lot 1."
    ),
    "metadata": {
        "document_type": "bid_response", "category": "Meat & Poultry",
        "bid_id": "BID-2024-118", "rfp_id": "RFP-2024-0489",
        "supplier_id": "SUP-1189", "supplier_name": "Northlake Protein Partners",
        "buyer_id": "ORG-RUSD", "bid_date": "2024-05-30",
        "total_bid_amount_usd": 3102700, "payment_terms": "Net-30",
        "status": "partially_awarded", "awarded_lots": ["LOT-0489-3"],
        "entities": ["BID-2024-118", "RFP-2024-0489", "SUP-1189", "ORG-RUSD",
                     "PER-SKELLEY", "NL-PLT-5010", "NL-PLT-5150", "NL-SEA-5900",
                     "FAC-NL-MODESTO", "CERT-NL-SQF2"],
        "relations": [
            ["SUP-1189", "SUBMITTED", "BID-2024-118"],
            ["BID-2024-118", "RESPONDS_TO", "RFP-2024-0489"],
            ["BID-2024-118", "BIDS_LOT", "LOT-0489-2"],
            ["BID-2024-118", "BIDS_LOT", "LOT-0489-3"],
            ["BID-2024-118", "COMPETES_WITH", "BID-2024-117"],
            ["BID-2024-118", "QUOTES_PRODUCT", "NL-PLT-5010"],
            ["PER-SKELLEY", "WORKS_FOR", "SUP-1189"],
            ["SUP-1189", "OPERATES_FACILITY", "FAC-NL-MODESTO"],
            ["SUP-1189", "HOLDS_CERTIFICATION", "CERT-NL-SQF2"],
        ],
    },
},

# ─────────────────────────── AWARD, CONTRACTS ───────────────────────────
"award-memo-rfp-0412": {
    "title": "Award Recommendation Memorandum — RFP-2024-0412 Fresh Produce Supply",
    "content": (
        "MEMORANDUM\n"
        "To: Board of Education, Riverside Unified School District\n"
        "From: Dana Whitfield (PER-DWHITFIELD), Director of Nutrition Services\n"
        "Prepared by: Miguel Arredondo (PER-MARREDONDO), Procurement Manager\n"
        "Date: 2024-05-20\n"
        "Subject: Award recommendation, RFP-2024-0412, Fresh Produce Supply, SY 2024-25\n\n"

        "1. RECOMMENDATION\n"
        "Staff recommend award of RFP-2024-0412 to Valle Verde Produce Co. (SUP-1042) under "
        "proposal BID-2024-089, in the amount of 1,742,880 USD for the initial contract year "
        "beginning 2024-08-01, with two optional one-year renewals. The resulting agreement is "
        "designated CTR-2024-0412-VV.\n\n"

        "2. PROPOSALS RECEIVED\n"
        "Two responsive proposals were received by the extended deadline of 2024-05-03:\n"
        "  BID-2024-089, Valle Verde Produce Co. (SUP-1042), 1,742,880 USD\n"
        "  BID-2024-091, Acme Foods Distribution (SUP-1077), 1,689,450 USD\n"
        "Four additional suppliers were solicited and did not respond.\n\n"

        "3. EVALUATION SCORES\n"
        "The evaluation committee comprised the Nutrition Services Director, the Procurement "
        "Manager, two site kitchen managers and a District food safety specialist. Scores are out "
        "of 100 on the weightings published in Section 3 of the solicitation.\n\n"
        "  Criterion                        Weight   Valle Verde   Acme Foods\n"
        "  Price                             40        36.8          40.0\n"
        "  Quality assurance & food safety   25        23.5          14.2\n"
        "  Delivery capability               20        18.6          13.5\n"
        "  Local sourcing                    10         8.5           5.4\n"
        "  MWBE participation                 5         4.0           1.6\n"
        "  TOTAL                            100        91.4          74.7\n\n"

        "4. BASIS OF RECOMMENDATION\n"
        "Acme Foods submitted the lower extended basket total, 53,430 USD below Valle Verde, and "
        "correctly received the full 40 price points. The recommendation nevertheless favours "
        "Valle Verde on three grounds.\n\n"
        "First, food safety. Section 4 of the solicitation requires a current SQF Level 2 "
        "certificate for every distribution centre servicing the contract on the proposal due "
        "date. Acme Foods' Riverside distribution centre certificate was in renewal and not "
        "current on 2024-05-03. Addendum No. 1, answer A2, stated explicitly that a certificate in "
        "renewal is not a current certificate for the purposes of this solicitation. Acme Foods' "
        "November 2023 audit also left one non-conformance open pending a capital repair. Acme "
        "scored 14.2 of 25 available points on this criterion, below the 15-point floor set in "
        "Section 3, which independently bars award on price advantage.\n\n"
        "Second, delivery. Acme Foods proposed a 60-day transition during which 9 outlying sites "
        "would receive deliveries between 8:30 AM and 9:30 AM, outside the window established by "
        "Addendum No. 1. Two committee members who manage site kitchens rated this unworkable "
        "against a 7:15 AM meal service.\n\n"
        "Third, local sourcing. Valle Verde committed 64% of annual spend within the 250-mile "
        "radius against Acme Foods' 41%, with named growers and quarterly reporting.\n\n"

        "5. FINANCIAL IMPACT\n"
        "The recommended award is 107,120 USD below the 1,850,000 USD estimated annual value in "
        "the solicitation. Funding is from the Cafeteria Special Revenue Fund. Cedar Valley "
        "Unified School District (ORG-CVUSD) has indicated intent to piggyback under the Central "
        "Valley Purchasing Cooperative at identical pricing, which does not alter the District's "
        "financial exposure.\n\n"

        "6. PROTEST\n"
        "No protest was filed within the five business day window closing 2024-05-28."
    ),
    "metadata": {
        "document_type": "award_memo", "category": "Produce",
        "rfp_id": "RFP-2024-0412", "buyer_id": "ORG-RUSD",
        "awarded_bid_id": "BID-2024-089", "awarded_supplier_id": "SUP-1042",
        "contract_id": "CTR-2024-0412-VV", "issue_date": "2024-05-20",
        "award_amount_usd": 1742880, "status": "approved",
        "entities": ["RFP-2024-0412", "BID-2024-089", "BID-2024-091", "SUP-1042", "SUP-1077",
                     "ORG-RUSD", "ORG-CVUSD", "CTR-2024-0412-VV",
                     "PER-DWHITFIELD", "PER-MARREDONDO"],
        "relations": [
            ["ORG-RUSD", "AWARDED", "BID-2024-089"],
            ["BID-2024-089", "SCORED", "91.4"],
            ["BID-2024-091", "SCORED", "74.7"],
            ["BID-2024-091", "LOST_TO", "BID-2024-089"],
            ["SUP-1042", "PARTY_TO", "CTR-2024-0412-VV"],
            ["ORG-RUSD", "PARTY_TO", "CTR-2024-0412-VV"],
            ["PER-DWHITFIELD", "WORKS_FOR", "ORG-RUSD"],
            ["PER-DWHITFIELD", "RECOMMENDED", "BID-2024-089"],
        ],
    },
},

"contract-ctr-2024-0412-vv": {
    "title": "Contract CTR-2024-0412-VV — Riverside USD and Valle Verde Produce Co.",
    "content": (
        "AGREEMENT FOR THE SUPPLY OF FRESH PRODUCE\n"
        "Contract number: CTR-2024-0412-VV\n"
        "Between: Riverside Unified School District (ORG-RUSD), 'the District'\n"
        "And: Valle Verde Produce Co. (SUP-1042), 'the Supplier'\n"
        "Arising from: RFP-2024-0412 and proposal BID-2024-089\n"
        "Effective: 2024-08-01. Initial term ends 2025-07-31. Two optional one-year renewals.\n"
        "Initial term value: 1,742,880 USD\n\n"

        "ARTICLE 1 — SCOPE\n"
        "The Supplier shall furnish all fresh fruits and vegetables listed in Attachment C to the "
        "42 school sites listed in Attachment A, at the unit prices set out in BID-2024-089, which "
        "are incorporated by reference and are firm-fixed for the initial term.\n\n"

        "ARTICLE 2 — PAYMENT\n"
        "Payment terms are Net-30 from the District's receipt of a correct invoice. The Supplier "
        "offers no early payment discount. The volume discount of 5% applies at the invoice level "
        "to any single site delivery exceeding 500 USD and shall be shown as a discount line on "
        "the invoice, not applied silently to unit prices. Minimum order value is 100 USD per site "
        "per delivery.\n\n"

        "ARTICLE 3 — DELIVERY\n"
        "Delivery shall occur twice weekly to every site, on Tuesdays and Thursdays, within the "
        "window 4:30 AM to 8:30 AM. Temperature-controlled product shall arrive between 34 and 41 "
        "degrees Fahrenheit. A delivery arriving outside the window without 24 hours prior written "
        "notice is recorded as a late delivery.\n\n"
        "ARTICLE 4 — PERFORMANCE STANDARDS AND LIQUIDATED DAMAGES\n"
        "4.1 The Supplier shall maintain a fill rate of not less than 97%, measured monthly by "
        "line item.\n"
        "4.2 The Supplier shall maintain an on-time delivery rate of not less than 95%, measured "
        "monthly by delivery event.\n"
        "4.3 Quality rejection rate shall remain below 2.0% of cases delivered, measured monthly.\n"
        "4.4 Where monthly on-time delivery falls below 92%, the District may assess liquidated "
        "damages of 2% of that month's invoiced value for each full percentage point below 92%, to "
        "a maximum of 10% of the monthly invoiced value. The parties agree this is a reasonable "
        "estimate of the District's costs of menu substitution and emergency procurement and is "
        "not a penalty.\n"
        "4.5 Two consecutive months below 92% on-time obliges the Supplier to submit a written "
        "corrective action plan within 10 business days.\n\n"

        "ARTICLE 5 — PRICE ADJUSTMENT\n"
        "No price adjustment is permitted during the initial term. At each renewal the Supplier "
        "may request adjustment once, capped at the lower of the twelve-month change in CPI-U "
        "food-at-home or 4%. Requests must be submitted no later than 60 days before the renewal "
        "date with supporting documentation.\n\n"

        "ARTICLE 6 — SUBSTITUTIONS\n"
        "No substitution may be delivered without prior written approval from the District's "
        "Nutrition Services Director (PER-DWHITFIELD). An unapproved substitution may be rejected "
        "at the Supplier's cost and is recorded as an unfilled line item.\n\n"

        "ARTICLE 7 — CERTIFICATIONS\n"
        "The Supplier shall maintain, for the full term, current GlobalG.A.P. certification and "
        "SQF Level 2 or higher certification for every facility servicing this contract, and shall "
        "notify the District in writing within 5 business days of any lapse, suspension or adverse "
        "audit finding. Failure to maintain certification is a material breach.\n\n"

        "ARTICLE 8 — LOCAL SOURCING\n"
        "The Supplier shall source not less than 64% of annual contract spend from growers whose "
        "primary production site lies within 250 miles of the District's central kitchen, and "
        "shall report actual local spend quarterly with grower addresses.\n\n"

        "ARTICLE 9 — TERMINATION\n"
        "The District may terminate for convenience on 60 days written notice, and for cause on "
        "30 days written notice where a material breach is not cured within that period. Loss of "
        "required certification, or three consecutive months below 92% on-time delivery, each "
        "constitute cause.\n\n"

        "ARTICLE 10 — COOPERATIVE PURCHASING\n"
        "Cedar Valley Unified School District (ORG-CVUSD) may purchase under this agreement "
        "through the Central Valley Purchasing Cooperative at identical unit prices and terms. Any "
        "such purchase is a separate obligation between the Supplier and ORG-CVUSD, and Riverside "
        "Unified School District bears no liability for it."
    ),
    "metadata": {
        "document_type": "contract", "category": "Produce",
        "contract_id": "CTR-2024-0412-VV", "rfp_id": "RFP-2024-0412",
        "bid_id": "BID-2024-089", "supplier_id": "SUP-1042",
        "supplier_name": "Valle Verde Produce Co.", "buyer_id": "ORG-RUSD",
        "effective_date": "2024-08-01", "expiry_date": "2025-07-31",
        "contract_value_usd": 1742880, "payment_terms": "Net-30", "status": "active",
        "entities": ["CTR-2024-0412-VV", "RFP-2024-0412", "BID-2024-089", "SUP-1042",
                     "ORG-RUSD", "ORG-CVUSD", "PER-DWHITFIELD"],
        "relations": [
            ["CTR-2024-0412-VV", "ARISES_FROM", "BID-2024-089"],
            ["CTR-2024-0412-VV", "ARISES_FROM", "RFP-2024-0412"],
            ["SUP-1042", "PARTY_TO", "CTR-2024-0412-VV"],
            ["ORG-RUSD", "PARTY_TO", "CTR-2024-0412-VV"],
            ["ORG-CVUSD", "MAY_PURCHASE_UNDER", "CTR-2024-0412-VV"],
            ["CTR-2024-0412-VV", "REQUIRES_CERTIFICATION", "CERT-VV-GAP"],
            ["CTR-2024-0412-VV", "REQUIRES_CERTIFICATION", "CERT-VV-SQF2"],
        ],
    },
},

"contract-ctr-2024-0489-tfh": {
    "title": "Contract CTR-2024-0489-TFH — Riverside USD and Tidewater Frozen Holdings",
    "content": (
        "AGREEMENT FOR THE SUPPLY OF FROZEN FOODS AND PROTEIN\n"
        "Contract number: CTR-2024-0489-TFH\n"
        "Between: Riverside Unified School District (ORG-RUSD), 'the District'\n"
        "And: Tidewater Frozen Holdings (SUP-1156), 'the Supplier'\n"
        "Arising from: RFP-2024-0489 and proposal BID-2024-117, Lots 1 and 2\n"
        "Effective: 2024-08-15. Initial term ends 2026-08-14, a 24-month term.\n"
        "Annual value: 2,984,220 USD\n\n"

        "ARTICLE 1 — SCOPE AND LOTS\n"
        "This agreement covers Lot 1, frozen prepared foods, frozen vegetables and frozen bakery, "
        "and Lot 2, frozen and further-processed protein including seafood. Lot 3, fresh and "
        "frozen poultry, was awarded separately to Northlake Protein Partners (SUP-1189) under "
        "contract CTR-2024-0489-NPP and is not covered here.\n\n"

        "ARTICLE 2 — AFFILIATE PERFORMANCE\n"
        "2.1 The Supplier has named Harbor Point Seafood (SUP-1157), its wholly owned subsidiary, "
        "as the fulfilment affiliate for all seafood line items under Lot 2.\n"
        "2.2 The Supplier remains fully and solely liable to the District for the performance of "
        "Harbor Point Seafood. The District will not accept a defence based on the acts or "
        "omissions of the affiliate.\n"
        "2.3 Harbor Point Seafood shall maintain Marine Stewardship Council Chain of Custody "
        "certification MSC-C-58817 for the full term. Lapse of that certification suspends the "
        "Supplier's right to deliver seafood line items until it is restored.\n\n"

        "ARTICLE 3 — PAYMENT\n"
        "Payment terms are Net-45 from receipt of a correct invoice. The District may take an "
        "early payment discount of 1.5% where payment issues within 15 days. The volume rebate of "
        "3% on monthly spend exceeding 200,000 USD is credited quarterly in arrears and shall be "
        "reconciled against actual invoiced spend. Minimum order value is 400 USD per delivery.\n\n"

        "ARTICLE 4 — DELIVERY AND TEMPERATURE\n"
        "Weekly delivery of frozen product to all 42 sites, arriving at 0 degrees Fahrenheit or "
        "below. Deliveries to the central kitchen shall use scheduled 45-minute dock slots. A load "
        "arriving above 10 degrees Fahrenheit is rejected in full at the Supplier's cost.\n\n"

        "ARTICLE 5 — PERFORMANCE STANDARDS\n"
        "5.1 Fill rate not less than 96%, measured monthly by line item.\n"
        "5.2 On-time delivery not less than 95%, measured monthly by delivery event.\n"
        "5.3 Quality rejection rate below 1.5% of cases delivered.\n"
        "5.4 A supplier falling below 92% on-time in any two consecutive months shall submit a "
        "corrective action plan within 10 business days. Failure below 90% in any single month "
        "entitles the District to withhold the quarterly volume rebate for that quarter.\n\n"

        "ARTICLE 6 — USDA FOODS\n"
        "The Supplier shall accept diverted USDA Foods entitlement beef and poultry, track "
        "entitlement value by district, and present the commodity offset as a separate line on "
        "every invoice. Unused entitlement value shall be reported to the District no later than "
        "45 days before the end of each entitlement year.\n\n"

        "ARTICLE 7 — TRACEABILITY\n"
        "The Supplier shall trace any delivered lot to its production facility and production date "
        "within four hours of a District request, and within two hours where the request arises "
        "from a suspected food safety incident or a public health enquiry.\n\n"

        "ARTICLE 8 — CERTIFICATIONS\n"
        "The Supplier shall maintain SQF Level 2 or higher for every facility touching product "
        "under this contract. The District notes that the Stockton facility certificate expires "
        "2025-04-30, within the contract term, and requires written evidence of renewal no later "
        "than 2025-05-14. Failure to provide evidence within that period suspends new orders until "
        "cured.\n\n"

        "ARTICLE 9 — PRICE ADJUSTMENT\n"
        "Prices are firm-fixed for the first 12 months. One adjustment is permitted effective "
        "2025-08-15, capped at the lower of the twelve-month change in the Producer Price Index "
        "for processed foods and feeds or 5%.\n\n"

        "ARTICLE 10 — TERMINATION\n"
        "Termination for convenience on 90 days written notice, reflecting the longer term and the "
        "Supplier's commodity processing commitments. Termination for cause on 30 days notice "
        "where a material breach is not cured. Loss of SQF certification at any facility servicing "
        "this contract is a material breach."
    ),
    "metadata": {
        "document_type": "contract", "category": "Frozen Foods",
        "contract_id": "CTR-2024-0489-TFH", "rfp_id": "RFP-2024-0489",
        "bid_id": "BID-2024-117", "supplier_id": "SUP-1156",
        "supplier_name": "Tidewater Frozen Holdings", "buyer_id": "ORG-RUSD",
        "effective_date": "2024-08-15", "expiry_date": "2026-08-14",
        "contract_value_usd": 2984220, "payment_terms": "Net-45", "status": "active",
        "entities": ["CTR-2024-0489-TFH", "RFP-2024-0489", "BID-2024-117", "SUP-1156",
                     "SUP-1157", "SUP-1189", "ORG-RUSD", "CTR-2024-0489-NPP",
                     "CERT-HP-MSC", "CERT-TW-SQF2-STK", "FAC-TW-STOCKTON"],
        "relations": [
            ["CTR-2024-0489-TFH", "ARISES_FROM", "BID-2024-117"],
            ["SUP-1156", "PARTY_TO", "CTR-2024-0489-TFH"],
            ["ORG-RUSD", "PARTY_TO", "CTR-2024-0489-TFH"],
            ["SUP-1157", "FULFILS_CATEGORY_UNDER", "CTR-2024-0489-TFH"],
            ["SUP-1156", "LIABLE_FOR", "SUP-1157"],
            ["SUP-1189", "PARTY_TO", "CTR-2024-0489-NPP"],
            ["CTR-2024-0489-TFH", "REQUIRES_CERTIFICATION", "CERT-HP-MSC"],
            ["CTR-2024-0489-TFH", "REQUIRES_CERTIFICATION", "CERT-TW-SQF2-STK"],
        ],
    },
},

# ─────────────────────── PERFORMANCE, PROFILE, COMPLIANCE ───────────────────────
"perf-review-q1-fy2025": {
    "title": "Q1 FY2025 Supplier Performance Review — Riverside USD Nutrition Services",
    "content": (
        "QUARTERLY SUPPLIER PERFORMANCE REVIEW\n"
        "Period: Q1 FY2025, 2024-09-01 through 2024-11-30\n"
        "Prepared by: Dana Whitfield (PER-DWHITFIELD), Director of Nutrition Services, "
        "Riverside Unified School District (ORG-RUSD)\n"
        "Issued: 2024-12-05\n\n"

        "1. SUMMARY\n"
        "Overall district satisfaction with nutrition services suppliers was 8.5 out of 10 for the "
        "quarter, measured by the District's site manager survey across 42 sites, up from 8.1 in "
        "Q4 FY2024. This is an aggregate district-level index and should not be confused with the "
        "individual supplier satisfaction ratings in Section 3, which are scored out of 5.\n\n"

        "2. SCORECARD\n"
        "  Supplier                       On-time   Fill rate   Quality rej.   Satisfaction\n"
        "  Valle Verde Produce Co.          96.2%      97.8%         0.9%          4.8/5\n"
        "  Globex Dairy Cooperative         98.4%      99.1%         0.4%          4.6/5\n"
        "  Summit Beverage Distributors     97.1%      96.9%         0.7%          4.5/5\n"
        "  Northlake Protein Partners       94.7%      96.0%         1.1%          4.3/5\n"
        "  Tidewater Frozen Holdings        92.0%      95.3%         1.7%          4.1/5\n\n"

        "3. SUPPLIER COMMENTARY\n\n"
        "3.1 Valle Verde Produce Co. (SUP-1042), contract CTR-2024-0412-VV. Customer satisfaction "
        "4.8 out of 5 stars, the highest of any supplier this quarter. On-time delivery 96.2% "
        "against a 95% contractual minimum. Fill rate 97.8% against a 97% minimum. Quality "
        "rejections 0.9% against a 2.0% ceiling. Credit memos totalled 4,120 USD, almost entirely "
        "from a single incident on 2024-10-17 when a romaine lettuce lot (SKU VV-PRD-1180) failed "
        "site-level inspection at nine sites. Valle Verde replaced the product within four hours "
        "and issued credit without dispute. Local sourcing reported at 66.1% against a 64% "
        "contractual commitment. No contractual remedies were triggered.\n\n"
        "3.2 Globex Dairy Cooperative (SUP-1103). The strongest operational performer of the "
        "quarter: on-time 98.4%, fill rate 99.1%, quality rejections 0.4%. Satisfaction 4.6 out of "
        "5. Zero loads rejected on temperature. The crate return programme removed an estimated "
        "12.4 tons of plastic during the quarter. Two site managers noted that the Friday delivery "
        "occasionally arrives at the end of the four-hour window, which compresses receiving.\n\n"
        "3.3 Summit Beverage Distributors (SUP-1204). On-time 97.1%, fill rate 96.9%, satisfaction "
        "4.5 out of 5. Performance is steady. No issues escalated.\n\n"
        "3.4 Northlake Protein Partners (SUP-1189), contract CTR-2024-0489-NPP, Lot 3 poultry. "
        "On-time 94.7%, marginally below the 95% contractual minimum, driven by three late "
        "deliveries in the week of 2024-10-28 attributed to a vehicle breakdown. Fill rate 96.0%. "
        "Satisfaction 4.3 out of 5. A written notice of the on-time shortfall was issued "
        "2024-12-05. No corrective action plan is required as the threshold for that remedy is two "
        "consecutive months below 92%.\n\n"
        "3.5 Tidewater Frozen Holdings (SUP-1156), contract CTR-2024-0489-TFH, Lots 1 and 2. The "
        "weakest performer this quarter and the only supplier requiring formal action. On-time "
        "delivery 92.0%, the lowest of any District supplier, against a 95% contractual minimum. "
        "Fill rate 95.3% against a 96% minimum. Quality rejections 1.7% against a 1.5% ceiling. "
        "Satisfaction 4.1 out of 5, the lowest recorded. October and November both fell below 92% "
        "on-time (91.6% and 91.8% respectively when measured monthly), which triggers Article 5.4 "
        "of CTR-2024-0489-TFH. A mandatory corrective action plan was requested on 2024-12-05 with "
        "a response due within 10 business days. Seafood line items fulfilled by Harbor Point "
        "Seafood (SUP-1157) performed better than the Tidewater average at 96.8% on-time, "
        "indicating the shortfall sits in the frozen prepared and vegetable categories rather than "
        "in the affiliate's seafood operation.\n\n"

        "4. ACTIONS\n"
        "4.1 Corrective action plan requested from Tidewater Frozen Holdings, due 2024-12-19.\n"
        "4.2 Written notice of on-time shortfall issued to Northlake Protein Partners.\n"
        "4.3 Globex Dairy Cooperative to be asked to review Friday route sequencing.\n"
        "4.4 The District will not assess liquidated damages this quarter. Article 4.4 of "
        "CTR-2024-0412-VV and Article 5.4 of CTR-2024-0489-TFH remain available if performance "
        "does not improve in Q2 FY2025."
    ),
    "metadata": {
        "document_type": "performance_review", "category": "Performance Review",
        "buyer_id": "ORG-RUSD", "period": "Q1 FY2025",
        "period_start": "2024-09-01", "period_end": "2024-11-30",
        "issue_date": "2024-12-05", "status": "issued",
        "district_satisfaction_index": "8.5/10",
        "entities": ["ORG-RUSD", "PER-DWHITFIELD", "SUP-1042", "SUP-1103", "SUP-1204",
                     "SUP-1189", "SUP-1156", "SUP-1157",
                     "CTR-2024-0412-VV", "CTR-2024-0489-TFH", "CTR-2024-0489-NPP",
                     "VV-PRD-1180"],
        "relations": [
            ["ORG-RUSD", "EVALUATED", "SUP-1042"],
            ["ORG-RUSD", "EVALUATED", "SUP-1103"],
            ["ORG-RUSD", "EVALUATED", "SUP-1156"],
            ["ORG-RUSD", "EVALUATED", "SUP-1189"],
            ["ORG-RUSD", "EVALUATED", "SUP-1204"],
            ["SUP-1042", "SATISFACTION_RATING", "4.8/5"],
            ["SUP-1103", "SATISFACTION_RATING", "4.6/5"],
            ["SUP-1204", "SATISFACTION_RATING", "4.5/5"],
            ["SUP-1189", "SATISFACTION_RATING", "4.3/5"],
            ["SUP-1156", "SATISFACTION_RATING", "4.1/5"],
            ["ORG-RUSD", "REQUESTED_CORRECTIVE_ACTION_FROM", "SUP-1156"],
            ["PER-DWHITFIELD", "AUTHORED", "perf-review-q1-fy2025"],
        ],
    },
},

"supplier-profile-tidewater": {
    "title": "Supplier Qualification Profile — Tidewater Frozen Holdings (SUP-1156)",
    "content": (
        "SUPPLIER QUALIFICATION PROFILE\n"
        "Supplier: Tidewater Frozen Holdings (SUP-1156)\n"
        "Maintained by: Riverside Unified School District Procurement\n"
        "Last updated: 2024-11-20\n\n"

        "1. CORPORATE\n"
        "Legal name: Tidewater Frozen Holdings, Inc. Founded 1987. Headquarters: 4400 Newton Road, "
        "Stockton, California. Ownership: privately held; majority stake acquired by Cordova "
        "Capital Partners in 2019. FY2024 revenue 412 million USD, up 6.2% year on year. "
        "Employees: 1,180. DUNS 08-441-9027.\n"
        "Categories served: Frozen Foods, Dry Groceries, Meat & Poultry. Customer base: 214 K-12 "
        "school districts, 46 higher education accounts, 31 healthcare accounts across California, "
        "Nevada and Arizona.\n\n"

        "2. SUBSIDIARY\n"
        "Harbor Point Seafood (SUP-1157) has been a wholly owned subsidiary since 2021-06-30, "
        "acquired for 34 million USD. Harbor Point operates independently from a 62,000 square "
        "foot facility in Oxnard, California, employs 145 people, and holds Marine Stewardship "
        "Council Chain of Custody certification MSC-C-58817 through 2026-01-31. Harbor Point is "
        "the named seafood fulfilment affiliate under BID-2024-117 and contract CTR-2024-0489-TFH. "
        "Under Article 2.2 of that contract, Tidewater remains solely liable to the District for "
        "Harbor Point's performance.\n\n"

        "3. FACILITIES\n"
        "  FAC-TW-STOCKTON      Stockton, CA      285,000 sq ft   SQF Level 2, exp. 2025-04-30\n"
        "                                                          BRCGS grade AA, exp. 2025-08-19\n"
        "  FAC-TW-BAKERSFIELD   Bakersfield, CA   140,000 sq ft   SQF Level 2, exp. 2025-10-08\n"
        "  FAC-HP-OXNARD        Oxnard, CA         62,000 sq ft   MSC CoC, exp. 2026-01-31\n"
        "Fleet: 96 refrigerated vehicles, average age 4.2 years, all fitted with continuous "
        "temperature telemetry.\n\n"

        "4. CONTRACT HISTORY WITH THE DISTRICT\n"
        "  CTR-2024-0489-TFH   Frozen Foods and Protein, Lots 1 and 2. Effective 2024-08-15 "
        "through 2026-08-14. Annual value 2,984,220 USD. Status: active.\n"
        "Tidewater was also solicited under RFP-2024-0412 (produce) and RFP-2024-0455 (dairy) and "
        "did not submit a proposal for either, citing category fit. Tidewater submitted question "
        "Q5 during the RFP-2024-0412 question period asking whether produce and frozen would be "
        "awarded together; the District confirmed they would not.\n\n"

        "5. PERFORMANCE HISTORY\n"
        "Q1 FY2025 was the supplier's first full quarter under CTR-2024-0489-TFH and the results "
        "were the weakest of any District supplier: on-time delivery 92.0%, fill rate 95.3%, "
        "quality rejections 1.7%, customer satisfaction 4.1 out of 5. Both October and November "
        "fell below the 92% monthly on-time threshold, triggering the mandatory corrective action "
        "plan under Article 5.4. The plan was requested 2024-12-05 with a response due 2024-12-19.\n"
        "Notably, seafood line items fulfilled by Harbor Point Seafood ran at 96.8% on-time, well "
        "above the Tidewater average, which locates the problem in frozen prepared foods and "
        "frozen vegetables rather than in the affiliate.\n\n"

        "6. RISK NOTES\n"
        "6.1 Certification timing. The Stockton SQF Level 2 certificate expires 2025-04-30, inside "
        "the contract term. Article 8 of CTR-2024-0489-TFH requires written evidence of renewal by "
        "2025-05-14 or new orders are suspended. The renewal audit is scheduled 2025-03-11. This "
        "is the District's most time-sensitive supplier compliance item.\n"
        "6.2 Audit findings. The September 2024 third-party audit at Stockton recorded two minor "
        "non-conformances, a freezer door seal and a gap in manual temperature logging. Both "
        "closed 2024-10-11 with auditor verification.\n"
        "6.3 Concentration. Tidewater represents 43% of the District's total SY 2024-25 food "
        "spend across all categories, the largest single-supplier concentration in the "
        "District's portfolio. Loss of this supplier mid-year would require emergency "
        "re-solicitation of two lots.\n\n"

        "7. REFERENCES\n"
        "Three K-12 references were checked in May 2024: Kern County Consolidated (positive, noted "
        "delivery timing issues in the first six months), Sierra Foothills USD (positive), and "
        "Antelope Valley Schools (positive, noted strong USDA Foods processing support)."
    ),
    "metadata": {
        "document_type": "supplier_profile", "category": "Frozen Foods",
        "supplier_id": "SUP-1156", "supplier_name": "Tidewater Frozen Holdings",
        "buyer_id": "ORG-RUSD", "issue_date": "2024-11-20", "status": "current",
        "entities": ["SUP-1156", "SUP-1157", "ORG-RUSD", "CTR-2024-0489-TFH",
                     "BID-2024-117", "RFP-2024-0412", "RFP-2024-0455", "RFP-2024-0489",
                     "FAC-TW-STOCKTON", "FAC-TW-BAKERSFIELD", "FAC-HP-OXNARD",
                     "CERT-TW-SQF2-STK", "CERT-TW-BRCGS", "CERT-HP-MSC", "PER-GHALLORAN"],
        "relations": [
            ["SUP-1157", "SUBSIDIARY_OF", "SUP-1156"],
            ["SUP-1156", "ACQUIRED", "SUP-1157"],
            ["SUP-1156", "OPERATES_FACILITY", "FAC-TW-STOCKTON"],
            ["SUP-1156", "OPERATES_FACILITY", "FAC-TW-BAKERSFIELD"],
            ["SUP-1157", "OPERATES_FACILITY", "FAC-HP-OXNARD"],
            ["FAC-TW-STOCKTON", "CERTIFIED_BY", "CERT-TW-SQF2-STK"],
            ["FAC-TW-STOCKTON", "CERTIFIED_BY", "CERT-TW-BRCGS"],
            ["FAC-HP-OXNARD", "CERTIFIED_BY", "CERT-HP-MSC"],
            ["SUP-1156", "PARTY_TO", "CTR-2024-0489-TFH"],
            ["SUP-1156", "DECLINED_TO_BID", "RFP-2024-0412"],
            ["SUP-1156", "DECLINED_TO_BID", "RFP-2024-0455"],
        ],
    },
},

"cert-compliance-register-2024": {
    "title": "Food Safety Certification and Compliance Register — Riverside USD, SY 2024-25",
    "content": (
        "FOOD SAFETY CERTIFICATION AND COMPLIANCE REGISTER\n"
        "Riverside Unified School District (ORG-RUSD), Nutrition Services\n"
        "Position as at 2024-12-01. Maintained by Miguel Arredondo (PER-MARREDONDO).\n\n"

        "1. PURPOSE\n"
        "This register tracks every certification the District's contracts require a supplier to "
        "hold, its expiry date, and the contractual consequence of lapse. It is reviewed monthly. "
        "Every District supply contract makes loss of a required certification a material breach.\n\n"

        "2. REGISTER\n\n"
        "CERT-VV-GAP — GlobalG.A.P., Valle Verde Produce Co. (SUP-1042)\n"
        "  Expires 2025-06-30. Required by Article 7, CTR-2024-0412-VV. Status: current.\n\n"
        "CERT-VV-SQF2 — SQF Level 2, Valle Verde Fresno facility (FAC-VV-FRESNO)\n"
        "  Expires 2025-09-14. Required by Article 7, CTR-2024-0412-VV. Status: current.\n"
        "  Last audit 2024-02-08, zero non-conformances.\n\n"
        "CERT-VV-ORG — USDA Organic handler, Valle Verde Produce Co. (SUP-1042)\n"
        "  Expires 2025-11-01. Not contractually required; supports organic line items. Current.\n\n"
        "CERT-GX-SQF3 — SQF Level 3, Globex Visalia plant (FAC-GX-VISALIA)\n"
        "  Expires 2026-02-28. Exceeds the SQF Level 2 minimum in RFP-2024-0455. Current.\n"
        "  Last audit 2024-03-05, zero non-conformances.\n\n"
        "CERT-GX-GRADEA — California Grade A dairy permit CA-DP-4471, Globex (SUP-1103)\n"
        "  Renewed annually, current through 2025-06-30. Mandatory for fluid milk supply.\n\n"
        "CERT-TW-SQF2-STK — SQF Level 2, Tidewater Stockton facility (FAC-TW-STOCKTON)\n"
        "  Expires 2025-04-30. ** EARLIEST EXPIRY IN THE REGISTER. ** Required by Article 8, "
        "CTR-2024-0489-TFH, which requires written evidence of renewal no later than 2025-05-14 or "
        "new orders are suspended until cured. Renewal audit scheduled 2025-03-11. Flagged for "
        "monthly follow-up from January 2025.\n"
        "  September 2024 audit: two minor non-conformances (freezer door seal; gap in manual "
        "temperature logging). Both closed 2024-10-11, auditor verified.\n\n"
        "CERT-TW-SQF2-BKF — SQF Level 2, Tidewater Bakersfield facility (FAC-TW-BAKERSFIELD)\n"
        "  Expires 2025-10-08. Required by Article 8, CTR-2024-0489-TFH. Status: current.\n\n"
        "CERT-TW-BRCGS — BRCGS grade AA, Tidewater Stockton facility (FAC-TW-STOCKTON)\n"
        "  Expires 2025-08-19. Not contractually required; supplementary assurance. Current.\n\n"
        "CERT-HP-MSC — Marine Stewardship Council Chain of Custody MSC-C-58817, "
        "Harbor Point Seafood (SUP-1157), Oxnard facility (FAC-HP-OXNARD)\n"
        "  Expires 2026-01-31. Required by Article 2.3, CTR-2024-0489-TFH. Lapse suspends the "
        "Supplier's right to deliver seafood line items until restored. Status: current.\n\n"
        "CERT-NL-SQF2 — SQF Level 2, Northlake Modesto facility (FAC-NL-MODESTO)\n"
        "  Expires 2025-12-05. Required by CTR-2024-0489-NPP. Status: current.\n"
        "  Last audit 2024-04-18, one minor non-conformance on label verification, closed "
        "2024-05-02.\n\n"
        "CERT-NL-USDA — USDA FSIS establishment EST. 18442, Northlake (SUP-1189)\n"
        "  Continuous inspection, no expiry. Mandatory for further-processed protein.\n\n"

        "3. SUPPLIERS NOT UNDER CONTRACT\n"
        "Acme Foods Distribution (SUP-1077) is not currently under District contract. Its Riverside "
        "facility (FAC-AF-RIV) SQF Level 2 certificate was in renewal and not current on "
        "2024-05-03, which was a material factor in the RFP-2024-0412 award decision. Its "
        "Sacramento facility (FAC-AF-SAC) certificate expires 2025-07-22. Should Acme Foods bid a "
        "future solicitation, both certificates must be current on the proposal due date.\n\n"

        "4. EXPIRY CALENDAR, NEXT 12 MONTHS\n"
        "  2025-04-30   CERT-TW-SQF2-STK    Tidewater Stockton      ** action required **\n"
        "  2025-06-30   CERT-VV-GAP         Valle Verde\n"
        "  2025-06-30   CERT-GX-GRADEA      Globex\n"
        "  2025-08-19   CERT-TW-BRCGS       Tidewater Stockton\n"
        "  2025-09-14   CERT-VV-SQF2        Valle Verde Fresno\n"
        "  2025-10-08   CERT-TW-SQF2-BKF    Tidewater Bakersfield\n"
        "  2025-11-01   CERT-VV-ORG         Valle Verde\n"
        "  2025-12-05   CERT-NL-SQF2        Northlake Modesto\n\n"

        "5. OPEN COMPLIANCE ITEMS\n"
        "5.1 Tidewater Frozen Holdings corrective action plan, requested 2024-12-05 following two "
        "consecutive months below 92% on-time delivery. Response due 2024-12-19. Performance, not "
        "certification, but tracked here because Article 10 of CTR-2024-0489-TFH makes sustained "
        "failure a termination-for-cause ground.\n"
        "5.2 No other open compliance items across the District's supplier portfolio."
    ),
    "metadata": {
        "document_type": "compliance_register", "category": "Compliance",
        "buyer_id": "ORG-RUSD", "issue_date": "2024-12-01", "status": "current",
        "entities": ["ORG-RUSD", "PER-MARREDONDO",
                     "SUP-1042", "SUP-1077", "SUP-1103", "SUP-1156", "SUP-1157", "SUP-1189",
                     "CERT-VV-GAP", "CERT-VV-SQF2", "CERT-VV-ORG", "CERT-GX-SQF3",
                     "CERT-GX-GRADEA", "CERT-TW-SQF2-STK", "CERT-TW-SQF2-BKF",
                     "CERT-TW-BRCGS", "CERT-HP-MSC", "CERT-NL-SQF2", "CERT-NL-USDA",
                     "FAC-VV-FRESNO", "FAC-GX-VISALIA", "FAC-TW-STOCKTON",
                     "FAC-TW-BAKERSFIELD", "FAC-HP-OXNARD", "FAC-NL-MODESTO",
                     "FAC-AF-RIV", "FAC-AF-SAC",
                     "CTR-2024-0412-VV", "CTR-2024-0489-TFH", "CTR-2024-0489-NPP"],
        "relations": [
            ["CERT-VV-GAP", "HELD_BY", "SUP-1042"],
            ["CERT-VV-SQF2", "COVERS_FACILITY", "FAC-VV-FRESNO"],
            ["CERT-GX-SQF3", "COVERS_FACILITY", "FAC-GX-VISALIA"],
            ["CERT-TW-SQF2-STK", "COVERS_FACILITY", "FAC-TW-STOCKTON"],
            ["CERT-TW-SQF2-BKF", "COVERS_FACILITY", "FAC-TW-BAKERSFIELD"],
            ["CERT-HP-MSC", "COVERS_FACILITY", "FAC-HP-OXNARD"],
            ["CERT-NL-SQF2", "COVERS_FACILITY", "FAC-NL-MODESTO"],
            ["CERT-TW-SQF2-STK", "REQUIRED_BY", "CTR-2024-0489-TFH"],
            ["CERT-HP-MSC", "REQUIRED_BY", "CTR-2024-0489-TFH"],
            ["CERT-VV-GAP", "REQUIRED_BY", "CTR-2024-0412-VV"],
            ["CERT-VV-SQF2", "REQUIRED_BY", "CTR-2024-0412-VV"],
            ["CERT-NL-SQF2", "REQUIRED_BY", "CTR-2024-0489-NPP"],
            ["PER-MARREDONDO", "MAINTAINS", "cert-compliance-register-2024"],
        ],
    },
},

"email-thread-tidewater-cap": {
    "title": "Email thread — Tidewater corrective action plan, December 2024",
    "content": (
        "EMAIL THREAD — District reference THR-2024-0912\n"
        "Subject: CTR-2024-0489-TFH — Corrective Action Plan request, Q1 FY2025\n"
        "Participants: Dana Whitfield (PER-DWHITFIELD, Riverside USD), Miguel Arredondo "
        "(PER-MARREDONDO, Riverside USD), Gregory Halloran (PER-GHALLORAN, Tidewater Frozen "
        "Holdings)\n\n"

        "----- Message 1 -----\n"
        "From: Dana Whitfield\n"
        "To: Gregory Halloran\n"
        "Cc: Miguel Arredondo\n"
        "Date: 2024-12-05 09:14 PT\n\n"
        "Gregory,\n"
        "Attached is the Q1 FY2025 performance review. Tidewater finished the quarter at 92.0% "
        "on-time delivery against the 95% minimum in Article 5.2 of CTR-2024-0489-TFH, with "
        "October at 91.6% and November at 91.8%. Two consecutive months below 92% triggers "
        "Article 5.4, so I am formally requesting a written corrective action plan. Please respond "
        "within 10 business days, by 2024-12-19.\n"
        "For context, fill rate was 95.3% against a 96% minimum and quality rejections were 1.7% "
        "against a 1.5% ceiling. Satisfaction came in at 4.1 out of 5, the lowest in our portfolio "
        "this quarter. I want to be clear that we are not assessing liquidated damages for Q1 and "
        "we are not contemplating termination. We do need a plan.\n"
        "Dana\n\n"

        "----- Message 2 -----\n"
        "From: Gregory Halloran\n"
        "To: Dana Whitfield\n"
        "Cc: Miguel Arredondo\n"
        "Date: 2024-12-09 16:41 PT\n\n"
        "Dana,\n"
        "Understood, and the numbers are not in dispute. Our own read matches yours. Three points "
        "while we prepare the formal plan.\n"
        "First, the shortfall is concentrated in Lot 1, frozen prepared and frozen vegetables, "
        "running out of Stockton. Our seafood line items through Harbor Point ran 96.8% on-time in "
        "the same period, so this is a Stockton routing problem rather than a network problem.\n"
        "Second, root cause. We added 19 new K-12 accounts in August and September and did not "
        "re-sequence the Stockton routes until late October. Riverside's 45-minute dock slots at "
        "the central kitchen sat at the end of a route that grew by roughly 70 minutes.\n"
        "Third, what we have already done. Route re-sequencing went live 2024-11-18 and November "
        "closed at 91.8% against October's 91.6%. December to date is tracking at 95.1%. We are "
        "adding two vehicles to the Stockton fleet in January.\n"
        "The written plan will be with you by 2024-12-17, ahead of your deadline.\n"
        "Gregory\n\n"

        "----- Message 3 -----\n"
        "From: Miguel Arredondo\n"
        "To: Gregory Halloran\n"
        "Cc: Dana Whitfield\n"
        "Date: 2024-12-10 08:02 PT\n\n"
        "Gregory,\n"
        "Thank you. Two additions for the plan, please.\n"
        "One, the quality rejection rate of 1.7% against the 1.5% ceiling needs addressing "
        "separately from delivery timing. Site managers logged most rejections against frozen "
        "vegetable cases with freezer burn, which reads as a cold chain or holding issue rather "
        "than a routing one.\n"
        "Two, unrelated to performance but time-sensitive: the SQF Level 2 certificate for "
        "Stockton (CERT-TW-SQF2-STK) expires 2025-04-30. Article 8 requires written evidence of "
        "renewal by 2025-05-14 or we suspend new orders. You have told us the renewal audit is "
        "booked for 2025-03-11. Please confirm that date in writing and send the certificate the "
        "day you receive it. This is on our register as the District's most time-sensitive "
        "compliance item.\n"
        "Miguel\n\n"

        "----- Message 4 -----\n"
        "From: Gregory Halloran\n"
        "To: Miguel Arredondo\n"
        "Cc: Dana Whitfield\n"
        "Date: 2024-12-10 11:27 PT\n\n"
        "Miguel,\n"
        "Confirmed on both. The Stockton SQF renewal audit is booked for 2025-03-11 with NSF as "
        "certification body; I will send the certificate the day it issues and will not wait for "
        "the 2025-05-14 deadline. On freezer burn, we pulled the affected lots and believe it "
        "traces to holding time in the Stockton freezer rather than transport, since trailer "
        "telemetry for those deliveries was clean. That analysis will be in the plan.\n"
        "Gregory"
    ),
    "metadata": {
        "document_type": "email_thread", "category": "Frozen Foods",
        "thread_id": "THR-2024-0912", "supplier_id": "SUP-1156",
        "supplier_name": "Tidewater Frozen Holdings", "buyer_id": "ORG-RUSD",
        "contract_id": "CTR-2024-0489-TFH",
        "issue_date": "2024-12-05", "status": "open",
        "entities": ["THR-2024-0912", "SUP-1156", "SUP-1157", "ORG-RUSD",
                     "PER-DWHITFIELD", "PER-MARREDONDO", "PER-GHALLORAN",
                     "CTR-2024-0489-TFH", "CERT-TW-SQF2-STK", "FAC-TW-STOCKTON"],
        "relations": [
            ["PER-DWHITFIELD", "SENT_MESSAGE_IN", "THR-2024-0912"],
            ["PER-MARREDONDO", "SENT_MESSAGE_IN", "THR-2024-0912"],
            ["PER-GHALLORAN", "SENT_MESSAGE_IN", "THR-2024-0912"],
            ["THR-2024-0912", "CONCERNS", "CTR-2024-0489-TFH"],
            ["THR-2024-0912", "CONCERNS", "CERT-TW-SQF2-STK"],
            ["ORG-RUSD", "REQUESTED_CORRECTIVE_ACTION_FROM", "SUP-1156"],
        ],
    },
},

}

rows = [(k, d["title"], d["content"], json.dumps(d["metadata"]),
         hashlib.sha256(d["content"].encode()).hexdigest(), None,
         spanner.COMMIT_TIMESTAMP)
        for k, d in SOURCE_DOCS.items()]
write("Documents",
      ["doc_id", "title", "content", "metadata", "content_hash", "indexed_hash", "updated_at"],
      rows)
print(f"Loaded {len(rows)} documents "
      f"(~{sum(len(d['content'].split()) for d in SOURCE_DOCS.values()):,} words).")

### 7b — Chunk, embed, store

`JSON` columns take a `JsonObject`, and the embedding goes in as a plain list of floats —
which is exactly why the column is `FLOAT64`.

In [ ]:
from google.cloud.spanner_v1 import JsonObject

def ingest(force=False):
    docs = query("""SELECT doc_id, title, content, metadata, content_hash, indexed_hash
                    FROM Documents
                    WHERE @force OR indexed_hash IS NULL OR indexed_hash != content_hash""",
                 params={"force": force}, types={"force": param_types.BOOL})
    if not docs:
        print("Nothing to ingest — everything is up to date.")
        return 0

    total = 0
    for d in docs:
        meta = d["metadata"] if isinstance(d["metadata"], dict) else json.loads(d["metadata"] or "{}")
        chunks = build_chunks(d["doc_id"], d["title"], d["content"], meta)
        # Embed the HEADER+TEXT version, not the clean text — that is what makes a bare
        # figure attributable to the right supplier.
        vecs = embed_texts([c["embed_input"] for c in chunks], "RETRIEVAL_DOCUMENT")

        write("Chunks",
              ["doc_id", "chunk_index", "chunk_id", "content", "embed_input",
               "token_count", "metadata", "embedding"],
              [(d["doc_id"], c["chunk_index"], f"{d['doc_id']}#{c['chunk_index']}",
                c["content"], c["embed_input"], c["token_count"],
                JsonObject(meta), list(v))
               for c, v in zip(chunks, vecs)])

        write("Documents", ["doc_id", "indexed_hash", "updated_at"],
              [(d["doc_id"], d["content_hash"], spanner.COMMIT_TIMESTAMP)])
        total += len(chunks)
        print(f"  {d['doc_id']:<32} {len(chunks):>2} chunks")

    print(f"\nIngested {len(docs)} documents, {total} chunks.")
    return total


ingest()
print(query_df("""SELECT count(*) AS chunks, count(DISTINCT doc_id) AS docs,
                         CAST(ROUND(AVG(token_count)) AS INT64) AS avg_tokens
                  FROM Chunks""").to_string(index=False))

## 8 — Hybrid retrieval, Spanner-style

One statement, two searches, fused with Reciprocal Rank Fusion — the same idea as the Postgres
version, expressed with Spanner's primitives:

| Leg | Postgres | Spanner |
|---|---|---|
| meaning | `embedding <=> qv` | `COSINE_DISTANCE(embedding, ...)` |
| keywords | `tsv @@ websearch_to_tsquery(...)` | `SEARCH(embed_tokens, ...)` + `SCORE(...)` |
| fusion | `FULL OUTER JOIN` + `1/(k+rank)` | `UNNEST(ARRAY(...)) WITH OFFSET` + `1/(k+rank)` |

`WITH OFFSET` is the trick: it hands you each row's position in an ordered array, which is
what RRF needs and what Spanner has instead of a window function over a search result.

The query vector goes in as an array literal rather than a parameter — it's numeric data, so
there is no injection surface, and it sidesteps float-type coercion between the client and an
`ARRAY<FLOAT64>` column.

In [ ]:
def _vec_literal(v):
    return "[" + ", ".join(repr(float(x)) for x in v) + "]"

def retrieve(question, qv=None, mode="hybrid", top_k=RERANK_CANDIDATES):
    """First-stage retrieval. mode: hybrid | vector | text."""
    if qv is None:
        qv = embed_query(question)
    vec_lit = _vec_literal(qv)
    vec_n = VECTOR_CANDIDATES if mode in ("hybrid", "vector") else 0
    txt_n = TEXT_CANDIDATES   if mode in ("hybrid", "text")   else 0

    sql = f"""
    WITH
    -- Vector leg: exact KNN. `WITH OFFSET AS rank` gives each hit its position.
    vec AS (
      SELECT x AS chunk_id, rank
      FROM UNNEST(ARRAY(
        SELECT chunk_id FROM Chunks
        WHERE embedding IS NOT NULL
        ORDER BY COSINE_DISTANCE(embedding, ARRAY<FLOAT64>{vec_lit})
        LIMIT {vec_n}
      )) AS x WITH OFFSET AS rank
    ),
    -- Keyword leg: SEARCH() filters, SCORE() ranks. Needs ChunksSearchIndex to exist.
    txt AS (
      SELECT x AS chunk_id, rank
      FROM UNNEST(ARRAY(
        SELECT chunk_id FROM Chunks
        WHERE SEARCH(embed_tokens, @q)
        ORDER BY SCORE(embed_tokens, @q) DESC
        LIMIT {txt_n}
      )) AS x WITH OFFSET AS rank
    ),
    fused AS (
      SELECT chunk_id, SUM(w) AS rrf_score,
             MIN(IF(leg = 'vec', rank, NULL)) AS vec_rank,
             MIN(IF(leg = 'txt', rank, NULL)) AS txt_rank
      FROM (
        SELECT chunk_id, 1.0 / ({RRF_K} + rank + 1) AS w, 'vec' AS leg FROM vec
        UNION ALL
        SELECT chunk_id, 1.0 / ({RRF_K} + rank + 1) AS w, 'txt' AS leg FROM txt
      )
      GROUP BY chunk_id
    )
    SELECT c.chunk_id, c.doc_id, c.chunk_index, c.content, c.metadata, d.title,
           f.rrf_score, f.vec_rank, f.txt_rank,
           COSINE_DISTANCE(c.embedding, ARRAY<FLOAT64>{vec_lit}) AS cosine_distance
    FROM fused f
    JOIN Chunks c ON c.chunk_id = f.chunk_id
    JOIN Documents d ON d.doc_id = c.doc_id
    ORDER BY f.rrf_score DESC, cosine_distance ASC
    LIMIT {top_k}
    """
    return query(sql, params={"q": question}, types={"q": param_types.STRING})


_q = "What was the customer satisfaction level for supplier performance review?"
for m in ("vector", "text", "hybrid"):
    print(f"\n--- {m} ---")
    for i, h in enumerate(retrieve(_q, mode=m, top_k=3), 1):
        print(f"{i}. {h['doc_id']:<30} vec={h['vec_rank']} txt={h['txt_rank']} "
              f"dist={h['cosine_distance']:.3f}")

## 9 — Build the graph

Two sources, neither of which needs an LLM:

**From document metadata** (below) — the `entities` / `relations` triples each document
carries.

**From your own tables** — if you have business tables in this database, every foreign key is
a relationship. `1.ipynb` has a full `inspect_schema_graph()` / `build_relational_graph()` pair
for that; the same logic works here by reading
`INFORMATION_SCHEMA.REFERENTIAL_CONSTRAINTS`. Skipped here to keep this file focused on the
graph itself.

Every relationship is written **twice** — forward and reverse — so traversal is always `->`.

In [ ]:
ID_PREFIX_TYPES = {
    "SUP": "supplier", "ORG": "organization", "PER": "person", "RFP": "rfp",
    "BID": "bid", "CTR": "contract", "CERT": "certification", "FAC": "facility",
    "ADD": "addendum", "LOT": "lot", "THR": "email_thread",
}
_SKU = re.compile(r"^[A-Z]{2}-[A-Z]{3}-\d{3,}$")

def _infer_type(nid):
    head = nid.split("-")[0].upper()
    if head in ID_PREFIX_TYPES: return ID_PREFIX_TYPES[head]
    if _SKU.match(nid):         return "product"
    # A relation object that is a value rather than a thing — "4.8/5", "Produce".
    # Kept as a node deliberately: it makes "who scored above 4.5" traversable.
    if len(nid) < 24 and not nid.isupper(): return "value"
    return "entity"


def build_graph_from_metadata():
    docs = query("SELECT doc_id, metadata FROM Documents")
    nodes, edges, labels = {}, set(), {}

    for d in docs:
        m = d["metadata"] if isinstance(d["metadata"], dict) else json.loads(d["metadata"] or "{}")
        # ids and names usually travel together in the same document — harvest the pairing
        # so nodes get real labels instead of bare ids.
        for k_id, k_name in (("supplier_id", "supplier_name"), ("buyer_id", "buyer_name")):
            if m.get(k_id) and m.get(k_name):
                labels[m[k_id]] = m[k_name]
        for e in m.get("entities") or []:
            nodes.setdefault(str(e), True)
        for tri in m.get("relations") or []:
            if isinstance(tri, (list, tuple)) and len(tri) == 3:
                s, p, o = (str(x) for x in tri)
                nodes.setdefault(s, True); nodes.setdefault(o, True)
                edges.add((s, p, o))

    if not nodes:
        print("No entities/relations in document metadata — nothing to build.")
        return 0, 0

    write("GraphNode", ["node_id", "node_type", "label", "degree", "properties", "source"],
          [(n, _infer_type(n), labels.get(n, n), 0, JsonObject({}), "metadata") for n in nodes])

    # Forward + reverse. Storing both directions means every traversal below is `->`,
    # which keeps the GQL to patterns that are documented with worked examples.
    edge_rows = []
    for s, p, o in edges:
        edge_rows.append((s, o, p, False, "metadata", JsonObject({})))
        edge_rows.append((o, s, p, True,  "metadata", JsonObject({})))
    write("GraphEdge", ["src_id", "dst_id", "predicate", "is_reverse", "source", "properties"],
          edge_rows)

    refresh_degrees()
    print(f"Graph: {len(nodes)} nodes, {len(edges)} relationships "
          f"({len(edge_rows)} rows incl. reverse).")
    return len(nodes), len(edges)


def refresh_degrees():
    """Cache each node's degree — the hub guard reads it during expansion."""
    def _txn(txn):
        txn.execute_update("""
            UPDATE GraphNode n
            SET degree = (SELECT COUNT(*) FROM GraphEdge e
                          WHERE e.src_id = n.node_id AND NOT e.is_reverse)
                       + (SELECT COUNT(*) FROM GraphEdge e
                          WHERE e.dst_id = n.node_id AND NOT e.is_reverse)
            WHERE TRUE""")
    database.run_in_transaction(_txn)


build_graph_from_metadata()
print(query_df("""SELECT node_type, COUNT(*) AS n FROM GraphNode
                  GROUP BY node_type ORDER BY n DESC LIMIT 10""").to_string(index=False))

## 10 — Link the graph to the text

`NodeMention` is how a question gets into the graph and how the graph gets back out to text.
Three methods, cheapest first: the document **declared** the entity, the chunk contains the
**id** verbatim, or the chunk contains the node's **name**.

Labels shorter than `MIN_LABEL_LEN` are skipped — linking on `"Milk"` would attach half the
corpus to one node and make expansion meaningless.

In [ ]:
MIN_LABEL_LEN = 6
LABEL_BATCH   = 800

def link_graph_to_chunks():
    chunks = query("SELECT chunk_id, doc_id, embed_input FROM Chunks")
    docs   = {d["doc_id"]: (d["metadata"] if isinstance(d["metadata"], dict)
                            else json.loads(d["metadata"] or "{}"))
              for d in query("SELECT doc_id, metadata FROM Documents")}
    nodes  = query("SELECT node_id, label FROM GraphNode")
    if not chunks:
        print("No chunks — run cell 7b first."); return 0

    known = {n["node_id"] for n in nodes}
    rows = {}
    def add(nid, c, method):
        if nid in known:
            rows.setdefault((nid, c["chunk_id"]), (c["doc_id"], method))

    for c in chunks:                                    # 1. declared in metadata
        for e in (docs.get(c["doc_id"], {}).get("entities") or []):
            add(str(e), c, "metadata")

    id_like = [n["node_id"] for n in nodes if re.match(r"^[A-Z]{2,5}-", n["node_id"])]
    for i in range(0, len(id_like), LABEL_BATCH):       # 2. the id appears verbatim
        pat = re.compile(r"(?<![A-Za-z0-9])(" +
                         "|".join(re.escape(x) for x in id_like[i:i+LABEL_BATCH]) +
                         r")(?![A-Za-z0-9-])")
        for c in chunks:
            for m in set(pat.findall(c["embed_input"])):
                add(m, c, "id_token")

    by_label = {(n["label"] or "").strip().lower(): n["node_id"] for n in nodes
                if len((n["label"] or "").strip()) >= MIN_LABEL_LEN
                and n["label"] != n["node_id"]}
    keys = list(by_label)
    for i in range(0, len(keys), LABEL_BATCH):         # 3. the name appears
        pat = re.compile(r"\b(" + "|".join(re.escape(x) for x in keys[i:i+LABEL_BATCH]) + r")\b",
                         re.IGNORECASE)
        for c in chunks:
            for m in set(pat.findall(c["embed_input"])):
                add(by_label[m.lower()], c, "label")

    write("NodeMention", ["node_id", "chunk_id", "doc_id", "method"],
          [(nid, cid, did, meth) for (nid, cid), (did, meth) in rows.items()])
    by_m = {}
    for _, (_, meth) in rows.items():
        by_m[meth] = by_m.get(meth, 0) + 1
    print("   by method:", ", ".join(f"{k}={v}" for k, v in sorted(by_m.items())))
    print(f"Linked {len(rows):,} node↔chunk mentions across {len(chunks):,} chunks.")
    return len(rows)


link_graph_to_chunks()
_s = query("""SELECT (SELECT COUNT(*) FROM GraphNode)   AS nodes,
                     (SELECT COUNT(*) FROM GraphEdge)   AS edges,
                     (SELECT COUNT(*) FROM NodeMention) AS mentions""")[0]
print(f"\nGRAPH: {_s['nodes']:,} nodes | {_s['edges']:,} edge rows | {_s['mentions']:,} mentions")

## 11 — Traversal, in GQL

**This is the cell that justifies the whole file.** Every query below starts with
`GRAPH ProcurementGraph` and uses graph patterns instead of joins.

Compare `path_between()` with the Postgres equivalent: `MATCH ANY SHORTEST (a)-[:Rel]->{1,5}(b)`
is one line. In Postgres it is a recursive CTE with a visited-path array, a cycle guard, and a
`MIN(hop)` aggregation — about forty lines, which is exactly what `1.ipynb` contains.

**Note two Spanner Graph rules the code works around:**

- *"Spanner Graph doesn't support returning graph elements as query results."* You cannot
  `RETURN n` — return its properties, or `SAFE_TO_JSON(n)`.
- Expansion is done **one hop per query**, in Python, rather than as a single `{1,2}` pattern.
  That is not a limitation — it is how the hub guard stays exact: after each hop we drop
  over-connected nodes *before* stepping again, so a node touching every document can be an
  answer but never a relay.

In [ ]:
def gql(body, params=None, types=None):
    """Run a GQL statement against the property graph."""
    return query(f"GRAPH {GRAPH_NAME}\n{body}", params, types)


def one_hop(node_ids, hub_max=None):
    """Every neighbour of these nodes, excluding relays through hubs."""
    if not node_ids:
        return []
    return gql("""
        MATCH (a:Entity)-[e:Rel]->(b:Entity)
        WHERE a.node_id IN UNNEST(@ids) AND a.degree <= @hub_max
        RETURN DISTINCT b.node_id AS node_id, b.label AS label,
               b.node_type AS node_type, b.degree AS degree
    """, params={"ids": list(node_ids),
                 "hub_max": GRAPH_HUB_DEGREE_MAX if hub_max is None else hub_max},
        types={"ids": param_types.Array(param_types.STRING),
               "hub_max": param_types.INT64})


def expand(seed_ids, hops=None, hub_max=None, limit=None):
    """Breadth-first expansion. Returns [{node_id, hop, node_type, label, degree}].

    One GQL query per hop, so the hub guard is applied between steps — exactly the
    semantics of the Postgres recursive CTE, but far easier to read.
    """
    hops  = GRAPH_HOPS if hops is None else hops
    limit = GRAPH_MAX_NODES if limit is None else limit
    seen, frontier = {}, list(dict.fromkeys(seed_ids))

    for s in gql("""MATCH (a:Entity) WHERE a.node_id IN UNNEST(@ids)
                    RETURN a.node_id AS node_id, a.label AS label,
                           a.node_type AS node_type, a.degree AS degree""",
                 params={"ids": frontier},
                 types={"ids": param_types.Array(param_types.STRING)}):
        seen[s["node_id"]] = dict(s, hop=0)

    for hop in range(1, hops + 1):
        nxt = []
        for r in one_hop(frontier, hub_max):
            if r["node_id"] not in seen:
                seen[r["node_id"]] = dict(r, hop=hop)
                nxt.append(r["node_id"])
        frontier = nxt
        if not frontier:
            break

    out = sorted(seen.values(), key=lambda r: (r["hop"], -(r["degree"] or 0)))
    return out[:limit]


def subgraph_facts(node_ids, limit=None):
    """Real edges whose BOTH endpoints are in the set — what the model is shown as facts."""
    if not node_ids:
        return []
    return gql("""
        MATCH (a:Entity)-[e:Rel]->(b:Entity)
        WHERE a.node_id IN UNNEST(@ids) AND b.node_id IN UNNEST(@ids)
          AND NOT e.is_reverse
        RETURN a.label AS src, a.node_id AS src_id, e.predicate AS predicate,
               b.label AS dst, b.node_id AS dst_id
        LIMIT @lim
    """, params={"ids": list(node_ids),
                 "lim": GRAPH_MAX_FACTS if limit is None else limit},
        types={"ids": param_types.Array(param_types.STRING), "lim": param_types.INT64})


def path_between(a_id, b_id, max_hops=5):
    """Shortest path — the query that is one line in GQL and forty in a recursive CTE."""
    rows = gql(f"""
        MATCH p = ANY SHORTEST (a:Entity {{node_id: @a}})-[e:Rel]->{{1,{max_hops}}}(b:Entity {{node_id: @b}})
        RETURN ARRAY_LENGTH(e) AS hops,
               ARRAY_TRANSFORM(NODES(p), n -> n.node_id) AS node_ids,
               ARRAY_TRANSFORM(e, x -> x.predicate)      AS predicates
    """, params={"a": a_id, "b": b_id},
        types={"a": param_types.STRING, "b": param_types.STRING})
    if not rows:
        print(f"No path from {a_id} to {b_id} within {max_hops} hops.")
        return None
    r = rows[0]
    print(f"{r['hops']} hop(s):")
    ids, preds = r["node_ids"], r["predicates"]
    for i, pr in enumerate(preds):
        print(f"   {ids[i]}  --[{pr}]-->  {ids[i+1]}")
    return r


def find_node(text, limit=15):
    rows = query("""SELECT node_id, node_type, label FROM GraphNode
                    WHERE LOWER(label) LIKE LOWER(@t) OR LOWER(node_id) LIKE LOWER(@t)
                    ORDER BY LENGTH(label) LIMIT @lim""",
                 params={"t": f"%{text}%", "lim": limit},
                 types={"t": param_types.STRING, "lim": param_types.INT64})
    for r in rows:
        print(f"  {r['node_type']:<16} {r['node_id']:<24} {r['label'][:50]}")
    return rows


def neighbors(node_id, hops=1):
    for r in expand([node_id], hops=hops):
        if r["hop"]:
            print(f"  {r['hop']} hop  {r['node_type']:<16} {r['node_id']:<24} {r['label'][:44]}")


def nodes_in_chunks(chunk_ids):
    """Which nodes do these chunks mention — the entry point into the graph."""
    if not chunk_ids:
        return []
    return query("""
        SELECT m.node_id, n.label, n.node_type, COUNT(*) AS hits
        FROM NodeMention m JOIN GraphNode n ON n.node_id = m.node_id
        WHERE m.chunk_id IN UNNEST(@ids)
        GROUP BY m.node_id, n.label, n.node_type ORDER BY hits DESC""",
        params={"ids": list(chunk_ids)},
        types={"ids": param_types.Array(param_types.STRING)})


def chunks_for_nodes(expanded, exclude_chunk_ids=(), limit=None, per_doc=None):
    """Back out of the graph into text.

    Two guards, both of which earned their place by failing without them:
      1. HOP WEIGHTING — 1/(1+hop)^2, so a near neighbour outranks a distant stranger.
      2. PER-DOCUMENT CAP — without it, one long document that name-drops thirty expanded
         entities eats the whole budget and the two-paragraph email holding the other half
         of the answer is never read.
    """
    if not expanded:
        return []
    ids = [n["node_id"] for n in expanded]
    ws  = [1.0 / (1 + n.get("hop", 0)) ** 2 for n in expanded]
    return query(f"""
        WITH w AS (
          SELECT node_id, weight
          FROM UNNEST(@ids) AS node_id WITH OFFSET o
          JOIN UNNEST(@ws)  AS weight  WITH OFFSET o2 ON o = o2
        ),
        scored AS (
          SELECT c.chunk_id, c.doc_id, c.chunk_index, c.content, c.metadata, d.title,
                 COUNT(DISTINCT m.node_id) AS node_hits, SUM(w.weight) AS graph_score
          FROM NodeMention m
          JOIN w          ON w.node_id  = m.node_id
          JOIN Chunks c   ON c.chunk_id = m.chunk_id
          JOIN Documents d ON d.doc_id  = c.doc_id
          WHERE c.chunk_id NOT IN UNNEST(@exclude)
          GROUP BY c.chunk_id, c.doc_id, c.chunk_index, c.content, c.metadata, d.title
        ),
        ranked AS (
          SELECT *, ROW_NUMBER() OVER (PARTITION BY doc_id
                                       ORDER BY graph_score DESC, chunk_id) AS rn
          FROM scored
        )
        SELECT chunk_id, doc_id, chunk_index, content, metadata, title, node_hits, graph_score
        FROM ranked WHERE rn <= @per_doc
        ORDER BY graph_score DESC, node_hits DESC LIMIT @lim
    """, params={"ids": ids, "ws": ws, "exclude": list(exclude_chunk_ids) or [""],
                 "per_doc": GRAPH_MAX_CHUNKS_PER_DOC if per_doc is None else per_doc,
                 "lim": GRAPH_MAX_EXTRA_CHUNKS if limit is None else limit},
        types={"ids": param_types.Array(param_types.STRING),
               "ws": param_types.Array(param_types.FLOAT64),
               "exclude": param_types.Array(param_types.STRING),
               "per_doc": param_types.INT64, "lim": param_types.INT64})


print("GQL traversal ready: expand(), neighbors(), path_between(), find_node().")

## 12 — Reranking and `ask_graphrag()`

Identical logic to `1.ipynb` — only the storage changed. Stage 3 is the point: **the graph
decides what else to read, not the embedding model.**

In [ ]:
REFUSAL = "I don't have that in the indexed documents."

RERANK_PROMPT = """You are a search relevance rater for a procurement document system.

Rate how well each passage answers the QUESTION, 0-10:
  10 = contains the exact answer
   7 = directly about the subject, partial answer
   4 = same topic, does not answer it
   0 = irrelevant

Judge each passage independently. Return ONLY a JSON array of objects with keys
"id" (the passage number) and "score" (integer 0-10). No prose.

QUESTION: {question}

PASSAGES:
{passages}"""

def rerank(question, hits, top_n=RERANK_TOP_N):
    if not hits:
        return hits
    passages = "\n\n".join(f"[{i}] ({h['title']}) {h['content'][:1200]}"
                            for i, h in enumerate(hits))
    try:
        r = _with_retry(lambda: genai_client.models.generate_content(
            model=RERANK_MODEL,
            contents=RERANK_PROMPT.format(question=question, passages=passages),
            config=GenerateContentConfig(temperature=0, response_mime_type="application/json")))
        scores = {int(x["id"]): float(x["score"]) for x in json.loads(r.text)}
    except Exception as e:
        print(f"  [rerank unavailable, keeping RRF order: {type(e).__name__}]")
        return hits[:top_n]
    for i, h in enumerate(hits):
        h["rerank_score"] = scores.get(i, 0.0)
    ranked = sorted(hits, key=lambda h: h["rerank_score"], reverse=True)
    return [h for h in ranked if h["rerank_score"] >= 4][:top_n] or ranked[:1]


GRAPH_SYSTEM_RULES = f"""You are a procurement analyst assistant. Answer using ONLY the
numbered SOURCES and the KNOWLEDGE GRAPH FACTS below.

Rules:
1. Cite every fact from a source with its number in square brackets, e.g. [2].
2. GRAPH FACTS are verified relationships from the database. Use them to connect entities
   across sources and to explain HOW things relate. Cite them as [graph].
3. If several entities or values legitimately match, list ALL of them with attribution.
   Never collapse distinct figures into one, and never present an aggregate as an
   individual's value or the reverse.
4. If the question names a specific entity, answer for THAT entity. If the sources cover a
   different one, say so rather than substituting it.
5. Quote figures exactly — same units, scale, currency and precision.
6. When the answer spans several documents, make the chain explicit.
7. If the sources and facts do not contain the answer, reply exactly: "{REFUSAL}"
8. Be concise. Lead with the answer, then the supporting detail."""


def format_sources(hits):
    out = []
    for i, h in enumerate(hits, 1):
        m = h.get("metadata") or {}
        if not isinstance(m, dict):
            m = json.loads(m or "{}")
        facets = " | ".join(f"{k}: {m[k]}" for k in HEADER_FIELDS if m.get(k))
        head = f"[{i}] {h['title']} (doc_id: {h['doc_id']}" + (f" | {facets}" if facets else "") + ")"
        out.append(f"{head}\n{h['content']}")
    return "\n\n".join(out)


def format_graph_facts(facts):
    if not facts:
        return "(no relationships found between the retrieved entities)"
    return "\n".join(f"  {f['src']} ({f['src_id']}) --[{f['predicate']}]--> {f['dst']} ({f['dst_id']})"
                     for f in facts)


def ask_graphrag(question, hops=None, verbose=False):
    """hybrid search → entity linking → GQL expansion → gather → rerank → generate."""
    T, t = {}, time.perf_counter

    t0 = t(); seed = retrieve(question, top_k=max(GRAPH_SEED_CHUNKS, RERANK_CANDIDATES // 2))
    T["seed_ms"] = (t() - t0) * 1000
    if not seed:
        return {"answer": REFUSAL, "sources": [],
                "graph": {"seed_nodes": [], "expanded": [], "facts": []},
                "timings": T, "refused": True}

    t0 = t(); seed_nodes = nodes_in_chunks([h["chunk_id"] for h in seed[:GRAPH_SEED_CHUNKS]])
    T["link_ms"] = (t() - t0) * 1000

    t0 = t(); expanded = expand([n["node_id"] for n in seed_nodes], hops=hops)
    T["expand_ms"] = (t() - t0) * 1000

    t0 = t()
    seen = {h["chunk_id"] for h in seed}
    extra = chunks_for_nodes(expanded, exclude_chunk_ids=seen)
    for e in extra:
        e["cosine_distance"] = 1.0; e["via_graph"] = True
    for h in seed:
        h["via_graph"] = False
    facts = subgraph_facts([n["node_id"] for n in expanded])
    T["gather_ms"] = (t() - t0) * 1000

    t0 = t(); hits = rerank(question, seed + extra); T["rerank_ms"] = (t() - t0) * 1000

    prompt = (f"{GRAPH_SYSTEM_RULES}\n\nKNOWLEDGE GRAPH FACTS:\n{format_graph_facts(facts)}"
              f"\n\nSOURCES:\n{format_sources(hits)}\n\nQUESTION: {question}")
    if verbose:
        print(prompt[:2500], "\n...\n")

    t0 = t()
    r = _with_retry(lambda: genai_client.models.generate_content(
        model=CHAT_MODEL, contents=prompt,
        config=GenerateContentConfig(temperature=0, max_output_tokens=1400)))
    T["generate_ms"] = (t() - t0) * 1000

    ans = (r.text or "").strip()
    return {"answer": ans,
            "sources": [{"n": i, "doc_id": h["doc_id"], "title": h["title"],
                         "content": h["content"], "rerank_score": h.get("rerank_score"),
                         "via_graph": h.get("via_graph", False)}
                        for i, h in enumerate(hits, 1)],
            "graph": {"seed_nodes": seed_nodes, "expanded": expanded, "facts": facts},
            "timings": T, "refused": ans.startswith(REFUSAL)}


def show_graph(res):
    print(res["answer"])
    g = res.get("graph", {})
    if g.get("seed_nodes"):
        print("\nEntities the question landed on:")
        for n in g["seed_nodes"][:8]:
            print(f"  • {n['node_type']:<14} {n['node_id']:<24} {n['label'][:42]}")
    if g.get("expanded"):
        by_hop = {}
        for n in g["expanded"]:
            by_hop.setdefault(n["hop"], []).append(n)
        print(f"\nGraph expansion ({len(g['expanded'])} nodes):")
        for hop in sorted(by_hop):
            names = ", ".join(n["label"][:24] for n in by_hop[hop][:6])
            more = f" (+{len(by_hop[hop])-6} more)" if len(by_hop[hop]) > 6 else ""
            print(f"  {hop} hop: {names}{more}")
    if g.get("facts"):
        print(f"\nRelationships given to the model ({len(g['facts'])}):")
        for f in g["facts"][:8]:
            print(f"  {f['src'][:26]} --[{f['predicate']}]--> {f['dst'][:26]}")
    print("\nSources:")
    for s in res["sources"]:
        tag = " ← pulled in by the graph" if s.get("via_graph") else ""
        print(f"  [{s['n']}] {s['doc_id']} (rerank={s['rerank_score']}){tag}")
    print("  timings:", {k: round(v) for k, v in res["timings"].items()}, "ms")


print("ask_graphrag() ready.")

## 13 — Try it

Look at the graph before you trust what it retrieves.

In [ ]:
find_node("Tidewater")

In [ ]:
# Two hops from the certificate that expires first.
neighbors("CERT-TW-SQF2-STK", hops=2)

In [ ]:
# Shortest path — one line of GQL. Compare with the recursive CTE in 1.ipynb.
path_between("SUP-1042", "CERT-HP-MSC", max_hops=5)

In [ ]:
# Raw GQL, straight from the docs' pattern vocabulary. Change it and see what happens.
query_df(f"""
GRAPH {GRAPH_NAME}
MATCH (a:Entity)-[e:Rel]->(b:Entity)
WHERE a.node_type = 'supplier' AND NOT e.is_reverse
RETURN a.label AS supplier, e.predicate AS relationship, b.node_id AS target
ORDER BY supplier
LIMIT 25
""")

In [ ]:
# GQL inside ordinary SQL, via GRAPH_TABLE — how you join graph results to normal tables.
query_df(f"""
SELECT gt.supplier, gt.cert_id, COUNT(m.chunk_id) AS chunks_mentioning
FROM GRAPH_TABLE(
  {GRAPH_NAME}
  MATCH (s:Entity)-[e:Rel]->(c:Entity)
  WHERE s.node_type = 'supplier' AND c.node_type = 'certification' AND NOT e.is_reverse
  RETURN s.label AS supplier, c.node_id AS cert_id
) AS gt
LEFT JOIN NodeMention m ON m.node_id = gt.cert_id
GROUP BY gt.supplier, gt.cert_id
ORDER BY chunks_mentioning DESC
""")

In [ ]:
# The multi-hop question. Watch the "← pulled in by the graph" tags: those are passages
# the embedding model would never have ranked highly, because they don't look like the question.
show_graph(ask_graphrag(
    "When does Tidewater's Stockton SQF certificate expire and what happens if it lapses?"))

In [ ]:
show_graph(ask_graphrag(
    "Which supplier is connected to Harbor Point Seafood, and what does that mean for the frozen contract?"))

## 14 — Notes, differences and teardown

### Things that will trip you up

| Symptom | Cause |
|---|---|
| `CREATE PROPERTY GRAPH` fails | database is PostgreSQL-dialect, or instance is Standard edition. Neither is fixable in place — recreate the database / upgrade the instance. |
| `SEARCH() ... requires a search index` | `ChunksSearchIndex` missing, or you searched a `TOKENLIST` held in a *different* search index. All token columns in one `SEARCH` must share one index. |
| Vector index never used | ANN needs all of: `APPROX_*` matching the index's `distance_type`, that call as the **sole** `ORDER BY` key, a `LIMIT`, and `WHERE embedding IS NOT NULL`. Pin it with `@{force_index=…}`. |
| `RETURN n` errors | *"Spanner Graph doesn't support returning graph elements as query results."* Return properties, or `SAFE_TO_JSON(n)`. |
| A property "exists" but errors outside `GRAPH_TABLE` | it must appear in the `RETURN` list to be visible to the outer SQL. |
| DDL seems not to have run | `update_ddl()` is async — `.result()` on it. `run_ddl()` here already does. |

### Differences from `1.ipynb` worth remembering

- **No graph DML.** GQL reads; you mutate `GraphNode` / `GraphEdge` with ordinary DML or
  mutations. That's why `build_graph_from_metadata()` uses `write()`.
- **No `ALTER PROPERTY GRAPH`.** Re-issue `CREATE OR REPLACE`.
- **Bounded quantifiers.** Every documented example bounds the hop count (`{1,3}`, `{2}`).
  Don't rely on an open-ended `{1,}` without testing it.
- **`ANY SHORTEST` is documented; `ALL SHORTEST` is not** — the openCypher comparison page
  says `allShortestPath` is unsupported. Stick to `ANY SHORTEST`.
- **Edges stored twice.** Halve your storage by dropping the reverse rows and switching the
  patterns to the undirected form `-[e:Rel]-`; that form is documented, but the *quantified*
  undirected form isn't shown with an example, so this file takes the safe road.

### Which file should you actually use?

Stay on **`1.ipynb` / Cloud SQL** while the graph is small — it costs nothing extra, and
Postgres traverses tens of thousands of edges without complaint. Move here when graph queries
become the workload rather than a helper: deep or variable-length paths, cheapest-path with a
cost function, or a graph too large for one machine. The node/edge model is identical, so the
migration is a data copy plus this schema.

### Teardown — do this, Spanner bills for provisioned capacity

```python
# database.drop()                                    # this database only
# instance.delete()                                  # the whole instance, and the bill
```

Or from the shell:
```bash
gcloud spanner databases delete procurement --instance=graphrag-demo
gcloud spanner instances delete graphrag-demo
```